# AnimalCLEF2026 Anatomy Template Registration v20260426

This is the new branch for the failure mode we identified visually: local keypoints and dots are only useful after the animal body is isolated, put into a comparable pose/scale, and interpreted with species-specific anatomy.

## Problem We Faced

The previous SAM-clean biometric branch fixed part of the data problem, but the matching was still too shallow:

- one global or semi-global descriptor asks whether two photos look similar overall;
- local SIFT/Harris-like points can still lock onto wrong regions if the body is not canonicalized first;
- SAM-clean images can be corrupted or incomplete, especially when a hand or occluding object was removed;
- the four species do not share image geometry, so one common matcher is the wrong assumption;
- using the 0.29758 submission as a merge scaffold can become a ceiling instead of a discovery method.

## Research Result Used Here

AnimalCLEF 2025 writeups showed that the largest jump came from adding segmentation and local matching/geometric verification on top of strong embeddings, not from backbone swaps alone. Patterned animal re-ID literature points in the same direction: compare local markings in spatially consistent coordinates, and allow partial matches when only part of the animal is visible.

For missing body parts, the key idea is not to hallucinate a full animal. The robust approach is a visibility-aware template: map the visible body into a canonical coordinate system, compare only mutually visible regions, and accept strong partial fingerprint matches only when their spatial pattern is consistent.

## Proposed Solution

```text
SAM-clean foreground
-> species-specific anatomy canonicalization
-> pattern map in template coordinates
-> masked overlap score + dot/spot constellation score
-> train-label validation where available
-> independent graph clustering, no current-best fallback
```

Species logic:

- `LynxID2025`: side-aware flank spot template. Opposite left/right flanks are penalized.
- `SalamanderID2025`: centerline skeleton unwrap into a straight body strip, then compare yellow/black body pattern under partial visibility.
- `SeaTurtleID2022`: head/scute-style square template, kept conservative because the existing branch is already strong.
- `TexasHornedLizards`: ventral belly oval template with dot constellation matching.

## Inputs

Attach these Kaggle inputs:

1. `animal-clef-2026`
2. `hanifnoerrofiq/animalclef2026-export-sam3-views-all-species-v2026`

No previous submission CSV is required for the main outputs. This notebook writes independent anatomy-template submissions.

CPU is enough. GPU is not used.
        

In [ ]:
from pathlib import Path

Path("species_fingerprint_final_swing_v20260426.py").write_text("from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport math\nimport random\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageFile\n\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nVERSION = \"species_fingerprint_final_swing_v20260426\"\nSEED = 20260426\nBACKGROUND_RGB = np.array([238, 238, 232], dtype=np.float32)\n\nSPECIES_ORDER = [\n    \"LynxID2025\",\n    \"SalamanderID2025\",\n    \"SeaTurtleID2022\",\n    \"TexasHornedLizards\",\n]\n\nCURRENT_BEST_FILENAME = \"submission_hybrid_v3rescue_p06_salamander_turtle_v20260425.csv\"\n\n\n@dataclass(frozen=True)\nclass EdgeRule:\n    name: str\n    min_inliers: int\n    min_inlier_ratio: float\n    min_score: float\n    allow_lynx_opposite_side: bool = False\n\n\n@dataclass(frozen=True)\nclass SpeciesConfig:\n    species: str\n    roi_kind: str\n    target_size: tuple[int, int]\n    top_k: int\n    max_keypoints: int\n    ratio_test: float\n    ransac_reproj: float\n    split_large_at: int\n    preserve_clusters_up_to: int\n    max_cluster_pair_size: int\n    conservative: EdgeRule\n    balanced: EdgeRule\n    swing: EdgeRule\n\n\nSPECIES_CONFIGS: dict[str, SpeciesConfig] = {\n    \"LynxID2025\": SpeciesConfig(\n        species=\"LynxID2025\",\n        roi_kind=\"lynx_flank_spots\",\n        target_size=(448, 288),\n        top_k=34,\n        max_keypoints=900,\n        ratio_test=0.78,\n        ransac_reproj=5.0,\n        split_large_at=42,\n        preserve_clusters_up_to=18,\n        max_cluster_pair_size=90,\n        conservative=EdgeRule(\"conservative\", min_inliers=24, min_inlier_ratio=0.22, min_score=0.42),\n        balanced=EdgeRule(\"balanced\", min_inliers=15, min_inlier_ratio=0.16, min_score=0.30),\n        swing=EdgeRule(\n            \"swing\",\n            min_inliers=9,\n            min_inlier_ratio=0.11,\n            min_score=0.20,\n            allow_lynx_opposite_side=False,\n        ),\n    ),\n    \"SalamanderID2025\": SpeciesConfig(\n        species=\"SalamanderID2025\",\n        roi_kind=\"salamander_straight_strip\",\n        target_size=(544, 176),\n        top_k=30,\n        max_keypoints=850,\n        ratio_test=0.80,\n        ransac_reproj=5.5,\n        split_large_at=9999,\n        preserve_clusters_up_to=9999,\n        max_cluster_pair_size=40,\n        conservative=EdgeRule(\"conservative\", min_inliers=18, min_inlier_ratio=0.18, min_score=0.36),\n        balanced=EdgeRule(\"balanced\", min_inliers=11, min_inlier_ratio=0.13, min_score=0.25),\n        swing=EdgeRule(\"swing\", min_inliers=7, min_inlier_ratio=0.09, min_score=0.18),\n    ),\n    \"SeaTurtleID2022\": SpeciesConfig(\n        species=\"SeaTurtleID2022\",\n        roi_kind=\"turtle_head_scutes\",\n        target_size=(416, 416),\n        top_k=22,\n        max_keypoints=900,\n        ratio_test=0.76,\n        ransac_reproj=4.5,\n        split_large_at=9999,\n        preserve_clusters_up_to=9999,\n        max_cluster_pair_size=35,\n        conservative=EdgeRule(\"conservative\", min_inliers=28, min_inlier_ratio=0.24, min_score=0.47),\n        balanced=EdgeRule(\"balanced\", min_inliers=22, min_inlier_ratio=0.20, min_score=0.39),\n        swing=EdgeRule(\"swing\", min_inliers=17, min_inlier_ratio=0.15, min_score=0.32),\n    ),\n    \"TexasHornedLizards\": SpeciesConfig(\n        species=\"TexasHornedLizards\",\n        roi_kind=\"texas_ventral_dots\",\n        target_size=(448, 320),\n        top_k=74,\n        max_keypoints=900,\n        ratio_test=0.78,\n        ransac_reproj=5.0,\n        split_large_at=20,\n        preserve_clusters_up_to=10,\n        max_cluster_pair_size=60,\n        conservative=EdgeRule(\"conservative\", min_inliers=18, min_inlier_ratio=0.26, min_score=0.34),\n        balanced=EdgeRule(\"balanced\", min_inliers=12, min_inlier_ratio=0.20, min_score=0.25),\n        swing=EdgeRule(\"swing\", min_inliers=8, min_inlier_ratio=0.16, min_score=0.18),\n    ),\n}\n\n\nOPTIONAL_SUBMISSION_FILENAMES = {\n    \"current_best\": CURRENT_BEST_FILENAME,\n    \"arwildfusion_v3_rescue_v2\": \"submission_sam_arwildfusion_v3_rescue_lbshape_v2.csv\",\n    \"p06_miewid_mega\": \"submission_p06_miewid_plus_mega_l384.csv\",\n    \"dualfoundation_species_hybrid\": \"submission_species_hybrid_v20260425.csv\",\n    \"dualfoundation_selected\": \"submission_selected_specieswise_hybrid.csv\",\n    \"specieswise_best_localari\": \"submission_specieswise_best_localari_hybrid_v20260425.csv\",\n}\n\n\nclass UnionFind:\n    def __init__(self, values: Iterable):\n        self.parent = {v: v for v in values}\n        self.size = {v: 1 for v in values}\n\n    def find(self, value):\n        if value not in self.parent:\n            self.parent[value] = value\n            self.size[value] = 1\n            return value\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a, b):\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return ra\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n        return ra\n\n\n@dataclass\nclass PatternItem:\n    image_id: int\n    row_idx: int\n    species: str\n    orientation: str\n    view_path: str\n    view_source: str\n    quality: float\n    keypoints: np.ndarray\n    descriptors: np.ndarray | None\n    vector: np.ndarray\n    fallback_vector: np.ndarray\n    part_vectors: np.ndarray\n    part_visibility: np.ndarray\n    fallback_part_vectors: np.ndarray\n    fallback_part_visibility: np.ndarray\n    visibility_coverage: float\n    n_keypoints: int\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            \"Final-swing species-specific fingerprint ensemble for AnimalCLEF2026. \"\n            \"Uses saved SAM views when present, extracts different local pattern ROIs \"\n            \"per species, verifies shortlisted pairs geometrically, and writes \"\n            \"conservative/balanced/high-swing hybrid submissions over the current best.\"\n        )\n    )\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--output-root\", type=str, default=None)\n    parser.add_argument(\"--sam-manifest\", type=str, default=None)\n    parser.add_argument(\"--current-best-submission\", type=str, default=None)\n    parser.add_argument(\"--extra-submission-root\", action=\"append\", default=[])\n    parser.add_argument(\"--species\", nargs=\"*\", default=SPECIES_ORDER)\n    parser.add_argument(\"--max-images-per-species\", type=int, default=0)\n    parser.add_argument(\"--max-side\", type=int, default=820)\n    parser.add_argument(\"--top-k-scale\", type=float, default=1.0)\n    parser.add_argument(\"--pair-budget-per-species\", type=int, default=65000)\n    parser.add_argument(\"--save-roi-preview\", action=\"store_true\")\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef find_data_root(user_value: str | None) -> Path:\n    candidates: list[Path] = []\n    if user_value:\n        candidates.append(Path(user_value))\n    candidates.extend(\n        [\n            Path(\"/kaggle/input/animal-clef-2026\"),\n            Path(\"/kaggle/input/competitions/animal-clef-2026\"),\n            Path.cwd() / \"animal-clef-2026\",\n            Path.cwd().parent / \"animal-clef-2026\",\n        ]\n    )\n    for root in candidates:\n        if (root / \"metadata.csv\").exists() and (root / \"sample_submission.csv\").exists():\n            return root.resolve()\n    raise FileNotFoundError(\"Could not locate AnimalCLEF2026 data root.\")\n\n\ndef find_sam_manifest(user_value: str | None) -> Path | None:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    direct = [\n        Path(\"/kaggle/working/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/reports/view_manifest_sam3_all_species.csv\"),\n        Path.cwd() / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n        Path.cwd().parent / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n    ]\n    for p in direct:\n        if p.exists():\n            return p.resolve()\n    for base in [Path(\"/kaggle/input\"), Path.cwd(), Path.cwd().parent]:\n        if base.exists():\n            try:\n                matches = list(base.rglob(\"view_manifest_sam3_all_species.csv\"))\n            except Exception:\n                matches = []\n            if matches:\n                return matches[0].resolve()\n    return None\n\n\ndef find_file_everywhere(filename: str, roots: Iterable[Path]) -> Path | None:\n    for root in roots:\n        candidate = root / filename\n        if candidate.exists():\n            return candidate.resolve()\n    for root in roots:\n        if not root.exists():\n            continue\n        try:\n            matches = list(root.rglob(filename))\n        except Exception:\n            matches = []\n        if matches:\n            matches.sort(key=lambda p: len(str(p)))\n            return matches[0].resolve()\n    return None\n\n\ndef find_submission_sources(args: argparse.Namespace, data_root: Path) -> dict[str, Path]:\n    roots: list[Path] = [\n        Path.cwd(),\n        Path.cwd().parent,\n        data_root.parent,\n        Path(\"/kaggle/input\"),\n    ]\n    roots.extend(Path(p) for p in args.extra_submission_root)\n    local_project = Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\")\n    roots.extend(\n        [\n            local_project / \"current_wildfusion_graph_v20260423\",\n            local_project / \"AnimalCLEF2026 SAM ARWildFusion v3_update\",\n            local_project / \"AnimalCLEF2026 SAM ARWildFusion v3_update\" / \"submissions\",\n            local_project / \"archive\" / \"AnimalCLEF2026 v4 Backbone Sweep + Non-EfficientNet Candidates\",\n            local_project / \"Masked Dual-Foundation Search v20260425 output\",\n        ]\n    )\n    roots = [p for p in roots if p is not None]\n\n    found: dict[str, Path] = {}\n    if args.current_best_submission:\n        p = Path(args.current_best_submission)\n        if p.exists():\n            found[\"current_best\"] = p.resolve()\n\n    for label, filename in OPTIONAL_SUBMISSION_FILENAMES.items():\n        if label in found:\n            continue\n        path = find_file_everywhere(filename, roots)\n        if path is not None:\n            found[label] = path\n\n    if \"current_best\" not in found:\n        raise FileNotFoundError(f\"Could not find {CURRENT_BEST_FILENAME}.\")\n    return found\n\n\ndef export_root_from_manifest(manifest_path: Path) -> Path:\n    if manifest_path.parent.name == \"reports\":\n        return manifest_path.parent.parent\n    return manifest_path.parent\n\n\ndef remap_export_path(path_value, export_root: Path | None) -> Path | None:\n    if path_value is None or pd.isna(path_value):\n        return None\n    s = str(path_value).strip()\n    if not s:\n        return None\n    p = Path(s)\n    if p.exists():\n        return p.resolve()\n    if export_root is None:\n        return None\n    normalized = s.replace(\"\\\\\", \"/\")\n    markers = [\n        \"animalclef_sam3_views_cache/\",\n        \"views/\",\n        \"mask_loose_square/\",\n        \"mask_full_square/\",\n    ]\n    for marker in markers:\n        if marker in normalized:\n            rel = normalized.split(marker, 1)[1]\n            if marker != \"animalclef_sam3_views_cache/\":\n                rel = marker + rel\n            candidate = export_root / Path(rel)\n            if candidate.exists():\n                return candidate.resolve()\n    return None\n\n\ndef prepare_test_metadata(data_root: Path, sam_manifest: Path | None) -> tuple[pd.DataFrame, dict]:\n    metadata = pd.read_csv(data_root / \"metadata.csv\").reset_index(drop=True)\n    if \"row_idx\" not in metadata.columns:\n        metadata[\"row_idx\"] = np.arange(len(metadata), dtype=np.int64)\n    if \"dataset\" in metadata.columns:\n        metadata[\"species_id\"] = metadata[\"dataset\"].astype(str)\n    else:\n        metadata[\"species_id\"] = metadata[\"path\"].str.replace(\"\\\\\", \"/\", regex=False).str.split(\"/\").str[1]\n    if \"split\" not in metadata.columns:\n        metadata[\"split\"] = np.where(metadata[\"path\"].str.contains(\"/test/\"), \"test\", \"train\")\n    if \"orientation\" not in metadata.columns:\n        metadata[\"orientation\"] = \"unknown\"\n    metadata[\"source_path\"] = metadata[\"path\"].map(lambda p: str(data_root / str(p)))\n\n    test = metadata[metadata[\"split\"].eq(\"test\")].copy()\n    test[\"view_path\"] = test[\"source_path\"]\n    test[\"view_source\"] = \"original\"\n    test[\"mask_path\"] = \"\"\n    test[\"mask_ok\"] = False\n\n    info = {\"manifest_path\": str(sam_manifest) if sam_manifest else None, \"manifest_rows\": 0, \"resolved_views\": 0}\n    if sam_manifest is None:\n        return test, info\n\n    manifest = pd.read_csv(sam_manifest)\n    info[\"manifest_rows\"] = int(len(manifest))\n    export_root = export_root_from_manifest(sam_manifest)\n    merge_key = \"row_idx\" if \"row_idx\" in manifest.columns else \"image_id\" if \"image_id\" in manifest.columns else None\n    if merge_key is None:\n        return test, info\n    merged = test.merge(manifest, on=merge_key, how=\"left\", suffixes=(\"\", \"_sam\"))\n\n    view_paths: list[str] = []\n    sam_view_paths: list[str] = []\n    mask_paths: list[str] = []\n    view_sources: list[str] = []\n    mask_ok: list[bool] = []\n    for row in merged.to_dict(\"records\"):\n        resolved_sam_view = None\n        for col in [\"loose_path\", \"mask_loose_path\", \"mask_full_path\", \"view_path\"]:\n            if col in row:\n                resolved_sam_view = remap_export_path(row.get(col), export_root)\n                if resolved_sam_view is not None:\n                    break\n        resolved_mask = None\n        for col in [\"mask_path\", \"binary_mask_path\"]:\n            if col in row:\n                resolved_mask = remap_export_path(row.get(col), export_root)\n                if resolved_mask is not None:\n                    break\n        # Corrected assumption: species is known and each image contains one\n        # individual, so the original image remains the pattern source. SAM is\n        # a guide for animal/body visibility, not the image to compare.\n        view_paths.append(str(Path(row[\"source_path\"])))\n        sam_view_paths.append(str(resolved_sam_view) if resolved_sam_view else \"\")\n        view_sources.append(\"original_sam_guided\" if resolved_mask is not None else \"original\")\n        mask_paths.append(str(resolved_mask) if resolved_mask else \"\")\n        mask_ok.append(resolved_mask is not None)\n    merged[\"view_path\"] = view_paths\n    merged[\"sam_view_path\"] = sam_view_paths\n    merged[\"view_source\"] = view_sources\n    merged[\"mask_path\"] = mask_paths\n    merged[\"mask_ok\"] = mask_ok\n    info[\"resolved_views\"] = int(sum(mask_ok))\n    return merged, info\n\n\ndef read_rgb(path: str | Path, max_side: int) -> np.ndarray:\n    img = Image.open(path).convert(\"RGB\")\n    w, h = img.size\n    scale = min(1.0, float(max_side) / float(max(w, h)))\n    if scale < 1.0:\n        img = img.resize((max(1, int(round(w * scale))), max(1, int(round(h * scale)))), Image.Resampling.BILINEAR)\n    return np.asarray(img)\n\n\ndef read_rgb_with_optional_mask(\n    image_path: str | Path,\n    mask_path: str | Path | None,\n    max_side: int,\n) -> tuple[np.ndarray, np.ndarray | None]:\n    img = Image.open(image_path).convert(\"RGB\")\n    w, h = img.size\n    mask_img = None\n    if mask_path:\n        p = Path(mask_path)\n        if p.exists():\n            try:\n                mask_img = Image.open(p).convert(\"L\")\n                if mask_img.size != img.size:\n                    mask_img = mask_img.resize(img.size, Image.Resampling.NEAREST)\n            except Exception:\n                mask_img = None\n    scale = min(1.0, float(max_side) / float(max(w, h)))\n    if scale < 1.0:\n        new_size = (max(1, int(round(w * scale))), max(1, int(round(h * scale))))\n        img = img.resize(new_size, Image.Resampling.BILINEAR)\n        if mask_img is not None:\n            mask_img = mask_img.resize(new_size, Image.Resampling.NEAREST)\n    rgb = np.asarray(img)\n    mask = None\n    if mask_img is not None:\n        mask = np.where(np.asarray(mask_img) > 127, 255, 0).astype(np.uint8)\n        if float(mask.mean() / 255.0) < 0.01:\n            mask = None\n    return rgb, mask\n\n\ndef estimate_foreground_mask(rgb: np.ndarray) -> np.ndarray:\n    arr = rgb.astype(np.float32)\n    h, w = arr.shape[:2]\n    border = np.concatenate(\n        [\n            arr[: max(2, h // 40), :, :].reshape(-1, 3),\n            arr[-max(2, h // 40) :, :, :].reshape(-1, 3),\n            arr[:, : max(2, w // 40), :].reshape(-1, 3),\n            arr[:, -max(2, w // 40) :, :].reshape(-1, 3),\n        ],\n        axis=0,\n    )\n    border_rgb = np.median(border, axis=0)\n    diff_border = np.linalg.norm(arr - border_rgb[None, None, :], axis=2)\n    diff_export_bg = np.linalg.norm(arr - BACKGROUND_RGB[None, None, :], axis=2)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    sat = hsv[:, :, 1].astype(np.float32)\n    mask = ((diff_export_bg > 20) & (diff_border > 8)) | ((diff_border > 28) & (sat > 18))\n    mask = mask.astype(np.uint8) * 255\n    kernel = np.ones((5, 5), np.uint8)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)\n    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)\n    if n_labels > 1:\n        biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n        mask = np.where(labels == biggest, 255, 0).astype(np.uint8)\n    coverage = float(mask.mean() / 255.0)\n    if coverage < 0.04 or coverage > 0.96:\n        mask = np.full((h, w), 255, dtype=np.uint8)\n    return mask\n\n\ndef bbox_from_mask(mask: np.ndarray, pad_frac: float = 0.05) -> tuple[int, int, int, int]:\n    ys, xs = np.where(mask > 0)\n    h, w = mask.shape[:2]\n    if len(xs) == 0:\n        return 0, 0, w, h\n    x1, x2 = int(xs.min()), int(xs.max()) + 1\n    y1, y2 = int(ys.min()), int(ys.max()) + 1\n    pad = int(round(max(x2 - x1, y2 - y1) * pad_frac))\n    return max(0, x1 - pad), max(0, y1 - pad), min(w, x2 + pad), min(h, y2 + pad)\n\n\ndef crop_to_mask(rgb: np.ndarray, mask: np.ndarray, pad_frac: float = 0.05) -> tuple[np.ndarray, np.ndarray]:\n    x1, y1, x2, y2 = bbox_from_mask(mask, pad_frac)\n    return rgb[y1:y2, x1:x2].copy(), mask[y1:y2, x1:x2].copy()\n\n\ndef pca_angle_degrees(mask: np.ndarray) -> float:\n    ys, xs = np.where(mask > 0)\n    if len(xs) < 30:\n        return 0.0\n    pts = np.stack([xs.astype(np.float32), ys.astype(np.float32)], axis=1)\n    pts -= pts.mean(axis=0, keepdims=True)\n    cov = np.cov(pts.T)\n    vals, vecs = np.linalg.eigh(cov)\n    vec = vecs[:, int(np.argmax(vals))]\n    return float(math.degrees(math.atan2(vec[1], vec[0])))\n\n\ndef rotate_bound(rgb: np.ndarray, mask: np.ndarray, angle: float) -> tuple[np.ndarray, np.ndarray]:\n    h, w = rgb.shape[:2]\n    center = (w / 2.0, h / 2.0)\n    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)\n    cos = abs(matrix[0, 0])\n    sin = abs(matrix[0, 1])\n    new_w = int((h * sin) + (w * cos))\n    new_h = int((h * cos) + (w * sin))\n    matrix[0, 2] += (new_w / 2.0) - center[0]\n    matrix[1, 2] += (new_h / 2.0) - center[1]\n    rgb_rot = cv2.warpAffine(\n        rgb,\n        matrix,\n        (new_w, new_h),\n        flags=cv2.INTER_LINEAR,\n        borderMode=cv2.BORDER_CONSTANT,\n        borderValue=(238, 238, 232),\n    )\n    mask_rot = cv2.warpAffine(\n        mask,\n        matrix,\n        (new_w, new_h),\n        flags=cv2.INTER_NEAREST,\n        borderMode=cv2.BORDER_CONSTANT,\n        borderValue=0,\n    )\n    return rgb_rot, mask_rot\n\n\ndef align_and_crop(rgb: np.ndarray, mask: np.ndarray, do_pca: bool = True) -> tuple[np.ndarray, np.ndarray]:\n    crop_rgb, crop_mask = crop_to_mask(rgb, mask, 0.08)\n    if do_pca:\n        angle = pca_angle_degrees(crop_mask)\n        if abs(angle) > 2:\n            crop_rgb, crop_mask = rotate_bound(crop_rgb, crop_mask, -angle)\n            crop_rgb, crop_mask = crop_to_mask(crop_rgb, crop_mask, 0.05)\n    return crop_rgb, crop_mask\n\n\ndef slice_roi(rgb: np.ndarray, mask: np.ndarray, box: tuple[float, float, float, float]) -> tuple[np.ndarray, np.ndarray]:\n    h, w = rgb.shape[:2]\n    x1 = int(round(w * box[0]))\n    y1 = int(round(h * box[1]))\n    x2 = int(round(w * box[2]))\n    y2 = int(round(h * box[3]))\n    x1, y1 = max(0, x1), max(0, y1)\n    x2, y2 = min(w, max(x1 + 1, x2)), min(h, max(y1 + 1, y2))\n    return rgb[y1:y2, x1:x2].copy(), mask[y1:y2, x1:x2].copy()\n\n\ndef resize_roi(rgb: np.ndarray, mask: np.ndarray, size: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:\n    w, h = size\n    rgb_resized = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_AREA)\n    mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)\n    mask_resized = np.where(mask_resized > 0, 255, 0).astype(np.uint8)\n    return rgb_resized, mask_resized\n\n\ndef species_roi(\n    rgb: np.ndarray,\n    species: str,\n    orientation: str,\n    config: SpeciesConfig,\n    mask_override: np.ndarray | None = None,\n) -> tuple[np.ndarray, np.ndarray, float]:\n    mask = mask_override if mask_override is not None else estimate_foreground_mask(rgb)\n    if mask.shape[:2] != rgb.shape[:2]:\n        mask = cv2.resize(mask, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)\n        mask = np.where(mask > 0, 255, 0).astype(np.uint8)\n    quality = 1.0\n    orientation = (orientation or \"unknown\").lower()\n    if config.roi_kind == \"turtle_head_scutes\":\n        aligned_rgb, aligned_mask = align_and_crop(rgb, mask, do_pca=False)\n        roi_rgb, roi_mask = slice_roi(aligned_rgb, aligned_mask, (0.04, 0.02, 0.96, 0.96))\n    else:\n        aligned_rgb, aligned_mask = align_and_crop(rgb, mask, do_pca=True)\n        if config.roi_kind == \"lynx_flank_spots\":\n            if orientation in {\"front\", \"back\", \"unknown\", \"\"}:\n                quality *= 0.72\n            roi_rgb, roi_mask = slice_roi(aligned_rgb, aligned_mask, (0.08, 0.14, 0.94, 0.88))\n        elif config.roi_kind == \"salamander_straight_strip\":\n            roi_rgb, roi_mask = slice_roi(aligned_rgb, aligned_mask, (0.03, 0.10, 0.97, 0.90))\n        elif config.roi_kind == \"texas_ventral_dots\":\n            roi_rgb, roi_mask = slice_roi(aligned_rgb, aligned_mask, (0.10, 0.08, 0.90, 0.94))\n        else:\n            roi_rgb, roi_mask = aligned_rgb, aligned_mask\n\n    roi_rgb, roi_mask = resize_roi(roi_rgb, roi_mask, config.target_size)\n    if config.roi_kind == \"texas_ventral_dots\":\n        h, w = roi_mask.shape[:2]\n        ellipse = np.zeros((h, w), dtype=np.uint8)\n        cv2.ellipse(\n            ellipse,\n            (w // 2, int(h * 0.54)),\n            (int(w * 0.38), int(h * 0.38)),\n            0,\n            0,\n            360,\n            255,\n            -1,\n        )\n        roi_mask = cv2.bitwise_and(roi_mask, ellipse)\n    if float(roi_mask.mean() / 255.0) < 0.05:\n        roi_mask = np.full(roi_mask.shape, 255, dtype=np.uint8)\n        quality *= 0.70\n    return roi_rgb, roi_mask, quality\n\n\ndef pattern_gray(rgb: np.ndarray, species: str) -> np.ndarray:\n    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    l = lab[:, :, 0]\n    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))\n    l_eq = clahe.apply(l)\n    if species in {\"LynxID2025\", \"TexasHornedLizards\"}:\n        dark = 255 - l_eq\n        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))\n        blackhat = cv2.morphologyEx(l_eq, cv2.MORPH_BLACKHAT, kernel)\n        gray = cv2.addWeighted(dark, 0.62, blackhat, 0.38, 0)\n    elif species == \"SalamanderID2025\":\n        yellow = lab[:, :, 2]\n        sat = hsv[:, :, 1]\n        gray = cv2.addWeighted(clahe.apply(yellow), 0.55, clahe.apply(sat), 0.45, 0)\n    else:\n        a = clahe.apply(lab[:, :, 1])\n        b = clahe.apply(lab[:, :, 2])\n        gray = cv2.addWeighted(l_eq, 0.60, cv2.addWeighted(a, 0.5, b, 0.5, 0), 0.40, 0)\n    gray = cv2.GaussianBlur(gray, (3, 3), 0)\n    return gray.astype(np.uint8)\n\n\ndef create_detector(max_keypoints: int):\n    try:\n        return \"sift\", cv2.SIFT_create(nfeatures=max_keypoints, contrastThreshold=0.015, edgeThreshold=12)\n    except Exception:\n        return \"orb\", cv2.ORB_create(nfeatures=max_keypoints, fastThreshold=7)\n\n\ndef normalize_vector(vec: np.ndarray) -> np.ndarray:\n    vec = vec.astype(np.float32, copy=False)\n    norm = float(np.linalg.norm(vec))\n    if norm > 0:\n        vec = vec / norm\n    return vec.astype(np.float32, copy=False)\n\n\ndef part_grid(config: SpeciesConfig) -> tuple[int, int]:\n    if config.roi_kind == \"salamander_straight_strip\":\n        return 3, 10\n    if config.roi_kind == \"turtle_head_scutes\":\n        return 4, 4\n    if config.roi_kind == \"texas_ventral_dots\":\n        return 5, 7\n    return 4, 7\n\n\ndef neutralize_missing(gray: np.ndarray, mask: np.ndarray | None) -> np.ndarray:\n    if mask is None:\n        return gray\n    valid = mask > 0\n    if valid.mean() < 0.02:\n        return gray\n    neutral = gray.copy()\n    fill = int(np.median(gray[valid]))\n    neutral[~valid] = fill\n    return neutral\n\n\ndef compute_vector(rgb: np.ndarray, gray: np.ndarray, mask: np.ndarray | None = None) -> np.ndarray:\n    gray_for_thumb = neutralize_missing(gray, mask)\n    small = cv2.resize(gray_for_thumb, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32).reshape(-1) / 255.0\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    hist_parts = []\n    for channel, bins, limit in [(0, 24, 180), (1, 16, 256), (2, 16, 256)]:\n        hist_mask = mask if mask is not None and float(mask.mean()) > 0 else None\n        hist = cv2.calcHist([hsv], [channel], hist_mask, [bins], [0, limit]).astype(np.float32).reshape(-1)\n        hist /= max(1e-6, float(hist.sum()))\n        hist_parts.append(hist)\n    vec = np.concatenate([small, *hist_parts]).astype(np.float32)\n    return normalize_vector(vec)\n\n\ndef compute_part_vectors(\n    rgb: np.ndarray,\n    gray: np.ndarray,\n    mask: np.ndarray,\n    config: SpeciesConfig,\n) -> tuple[np.ndarray, np.ndarray, float]:\n    rows, cols = part_grid(config)\n    h, w = gray.shape[:2]\n    vectors: list[np.ndarray] = []\n    visibility: list[bool] = []\n    total_valid = float((mask > 0).mean())\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    for gy in range(rows):\n        y1 = int(round(h * gy / rows))\n        y2 = int(round(h * (gy + 1) / rows))\n        for gx in range(cols):\n            x1 = int(round(w * gx / cols))\n            x2 = int(round(w * (gx + 1) / cols))\n            cell_mask = mask[y1:y2, x1:x2]\n            cell_gray = gray[y1:y2, x1:x2]\n            cell_rgb = rgb[y1:y2, x1:x2]\n            cell_hsv = hsv[y1:y2, x1:x2]\n            coverage = float((cell_mask > 0).mean()) if cell_mask.size else 0.0\n            visible = coverage >= 0.12\n            visibility.append(visible)\n            if not visible:\n                vectors.append(np.zeros(64 + 32, dtype=np.float32))\n                continue\n            valid = cell_mask > 0\n            cell_neutral = cell_gray.copy()\n            cell_neutral[~valid] = int(np.median(cell_gray[valid]))\n            thumb = cv2.resize(cell_neutral, (8, 8), interpolation=cv2.INTER_AREA).astype(np.float32).reshape(-1) / 255.0\n            hist_parts = []\n            for channel, bins, limit in [(0, 12, 180), (1, 10, 256), (2, 10, 256)]:\n                hist = cv2.calcHist([cell_hsv], [channel], cell_mask, [bins], [0, limit]).astype(np.float32).reshape(-1)\n                hist /= max(1e-6, float(hist.sum()))\n                hist_parts.append(hist)\n            vectors.append(normalize_vector(np.concatenate([thumb, *hist_parts]).astype(np.float32)))\n    return np.stack(vectors).astype(np.float32), np.asarray(visibility, dtype=bool), total_valid\n\n\ndef part_similarity_from_arrays(\n    vec_a: np.ndarray,\n    vis_a: np.ndarray,\n    vec_b: np.ndarray,\n    vis_b: np.ndarray,\n) -> tuple[float, float, int]:\n    if vec_a.size == 0 or vec_b.size == 0:\n        return 0.0, 0.0, 0\n    common = vis_a & vis_b\n    n_common = int(common.sum())\n    n_possible = int((vis_a | vis_b).sum())\n    if n_common == 0:\n        return 0.0, 0.0, 0\n    sims = np.sum(vec_a[common] * vec_b[common], axis=1)\n    sims = np.clip(sims, -1.0, 1.0)\n    if len(sims) >= 4:\n        # Partial matching: emphasize the best mutually visible zones instead\n        # of punishing a crop where SAM removed a limb/body patch.\n        keep = max(2, int(math.ceil(len(sims) * 0.70)))\n        score = float(np.mean(np.sort(sims)[-keep:]))\n    else:\n        score = float(np.mean(sims))\n    overlap = float(n_common / max(1, n_possible))\n    return score, overlap, n_common\n\n\ndef occlusion_aware_similarity(a: PatternItem, b: PatternItem) -> tuple[float, float, int, float]:\n    candidates = [\n        part_similarity_from_arrays(a.part_vectors, a.part_visibility, b.part_vectors, b.part_visibility),\n        part_similarity_from_arrays(\n            a.fallback_part_vectors,\n            a.fallback_part_visibility,\n            b.fallback_part_vectors,\n            b.fallback_part_visibility,\n        ),\n        part_similarity_from_arrays(a.part_vectors, a.part_visibility, b.fallback_part_vectors, b.fallback_part_visibility),\n        part_similarity_from_arrays(a.fallback_part_vectors, a.fallback_part_visibility, b.part_vectors, b.part_visibility),\n    ]\n    part_score, part_overlap, part_cells = max(candidates, key=lambda x: (x[0], x[2]))\n    global_primary = float(np.dot(a.vector, b.vector))\n    global_fallback = float(np.dot(a.fallback_vector, b.fallback_vector))\n    visual_sim = max(global_primary, global_fallback, part_score)\n    return visual_sim, float(part_score), int(part_cells), float(part_overlap)\n\n\ndef extract_pattern_item(row: dict, config: SpeciesConfig, max_side: int, detector_name: str, detector) -> PatternItem:\n    mask_path = str(row.get(\"mask_path\", \"\")).strip()\n    rgb, sam_mask = read_rgb_with_optional_mask(row[\"view_path\"], mask_path, max_side=max_side)\n    roi_rgb, roi_mask, quality = species_roi(\n        rgb,\n        row[\"species_id\"],\n        str(row.get(\"orientation\", \"unknown\")),\n        config,\n        mask_override=sam_mask,\n    )\n    gray = pattern_gray(roi_rgb, row[\"species_id\"])\n    kps, desc = detector.detectAndCompute(gray, roi_mask)\n    if kps is None:\n        kps = []\n    pts = np.array([kp.pt for kp in kps], dtype=np.float32).reshape(-1, 2)\n    if desc is not None and len(desc) > 0:\n        if detector_name == \"sift\":\n            desc = desc.astype(np.float32)\n            desc /= np.maximum(1e-7, desc.sum(axis=1, keepdims=True))\n            desc = np.sqrt(desc)\n        else:\n            desc = desc.astype(np.uint8)\n    else:\n        desc = None\n    vec = compute_vector(roi_rgb, gray, roi_mask)\n    part_vecs, part_vis, coverage = compute_part_vectors(roi_rgb, gray, roi_mask, config)\n    fallback_vec = vec\n    fallback_part_vecs = part_vecs\n    fallback_part_vis = part_vis\n    source_path = str(row.get(\"source_path\", \"\")).strip()\n    # Fallback descriptor deliberately ignores SAM if present. It helps\n    # shortlist candidates when SAM masks are too conservative, while the\n    # primary descriptor still uses the SAM-guided animal/body region.\n    if source_path and Path(source_path).exists() and sam_mask is not None:\n        try:\n            original_rgb = read_rgb(source_path, max_side=max_side)\n            original_roi_rgb, original_roi_mask, original_quality = species_roi(\n                original_rgb,\n                row[\"species_id\"],\n                str(row.get(\"orientation\", \"unknown\")),\n                config,\n                mask_override=None,\n            )\n            original_gray = pattern_gray(original_roi_rgb, row[\"species_id\"])\n            fallback_vec = compute_vector(original_roi_rgb, original_gray, original_roi_mask)\n            fallback_part_vecs, fallback_part_vis, original_coverage = compute_part_vectors(\n                original_roi_rgb,\n                original_gray,\n                original_roi_mask,\n                config,\n            )\n            quality = min(quality, 0.75 + 0.25 * original_quality)\n            coverage = max(coverage, min(0.95, original_coverage))\n        except Exception:\n            fallback_vec = vec\n            fallback_part_vecs = part_vecs\n            fallback_part_vis = part_vis\n    if coverage < 0.22 and row[\"species_id\"] != \"SeaTurtleID2022\":\n        quality *= 0.78\n    combined_vec = normalize_vector(0.72 * vec + 0.28 * fallback_vec)\n    return PatternItem(\n        image_id=int(row[\"image_id\"]),\n        row_idx=int(row.get(\"row_idx\", row[\"image_id\"])),\n        species=str(row[\"species_id\"]),\n        orientation=str(row.get(\"orientation\", \"unknown\")).lower(),\n        view_path=str(row[\"view_path\"]),\n        view_source=str(row.get(\"view_source\", \"original\")),\n        quality=float(quality),\n        keypoints=pts,\n        descriptors=desc,\n        vector=combined_vec,\n        fallback_vector=fallback_vec,\n        part_vectors=part_vecs,\n        part_visibility=part_vis,\n        fallback_part_vectors=fallback_part_vecs,\n        fallback_part_visibility=fallback_part_vis,\n        visibility_coverage=float(coverage),\n        n_keypoints=len(kps),\n    )\n\n\ndef save_roi_preview(rows: list[dict], out_path: Path, config: SpeciesConfig, max_side: int, limit: int = 24) -> None:\n    if not rows:\n        return\n    thumbs = []\n    for row in rows[:limit]:\n        try:\n            mask_path = str(row.get(\"mask_path\", \"\")).strip()\n            rgb, sam_mask = read_rgb_with_optional_mask(row[\"view_path\"], mask_path, max_side=max_side)\n            roi_rgb, roi_mask, _ = species_roi(\n                rgb,\n                row[\"species_id\"],\n                str(row.get(\"orientation\", \"unknown\")),\n                config,\n                mask_override=sam_mask,\n            )\n            overlay = roi_rgb.copy()\n            overlay[roi_mask == 0] = (overlay[roi_mask == 0] * 0.35 + np.array([238, 238, 232]) * 0.65).astype(np.uint8)\n            thumbs.append(Image.fromarray(overlay).resize((160, 112)))\n        except Exception:\n            continue\n    if not thumbs:\n        return\n    cols = 4\n    rows_n = int(math.ceil(len(thumbs) / cols))\n    canvas = Image.new(\"RGB\", (cols * 160, rows_n * 112), (238, 238, 232))\n    for idx, thumb in enumerate(thumbs):\n        canvas.paste(thumb, ((idx % cols) * 160, (idx // cols) * 112))\n    canvas.save(out_path, quality=90)\n\n\ndef load_submission(path: Path) -> pd.DataFrame:\n    df = pd.read_csv(path)\n    if \"image_id\" not in df.columns or \"cluster\" not in df.columns:\n        raise ValueError(f\"{path} must contain image_id and cluster columns.\")\n    df = df[[\"image_id\", \"cluster\"]].copy()\n    df[\"image_id\"] = df[\"image_id\"].astype(int)\n    df[\"cluster\"] = df[\"cluster\"].astype(str)\n    return df\n\n\ndef labels_for_species(submission: pd.DataFrame, species_rows: pd.DataFrame) -> dict[int, str]:\n    merged = species_rows[[\"image_id\"]].merge(submission, on=\"image_id\", how=\"left\")\n    if merged[\"cluster\"].isna().any():\n        missing = merged.loc[merged[\"cluster\"].isna(), \"image_id\"].head().tolist()\n        raise ValueError(f\"Submission is missing image ids: {missing}\")\n    return {int(r.image_id): str(r.cluster) for r in merged.itertuples(index=False)}\n\n\ndef pair_key(a: int, b: int) -> tuple[int, int]:\n    return (a, b) if a < b else (b, a)\n\n\ndef cluster_pair_votes(\n    source_submissions: dict[str, pd.DataFrame],\n    species_rows: pd.DataFrame,\n    config: SpeciesConfig,\n) -> dict[tuple[int, int], int]:\n    votes: dict[tuple[int, int], int] = {}\n    for source_name, sub in source_submissions.items():\n        if source_name == \"current_best\":\n            continue\n        labels = labels_for_species(sub, species_rows)\n        groups: dict[str, list[int]] = {}\n        for image_id, cluster in labels.items():\n            groups.setdefault(cluster, []).append(image_id)\n        for members in groups.values():\n            if 1 < len(members) <= config.max_cluster_pair_size:\n                for a, b in itertools.combinations(sorted(members), 2):\n                    key = pair_key(a, b)\n                    votes[key] = votes.get(key, 0) + 1\n    return votes\n\n\ndef current_cluster_pairs(labels: dict[int, str], max_cluster_pair_size: int) -> set[tuple[int, int]]:\n    groups: dict[str, list[int]] = {}\n    for image_id, cluster in labels.items():\n        groups.setdefault(cluster, []).append(image_id)\n    pairs: set[tuple[int, int]] = set()\n    for members in groups.values():\n        if 1 < len(members) <= max_cluster_pair_size:\n            for a, b in itertools.combinations(sorted(members), 2):\n                pairs.add(pair_key(a, b))\n    return pairs\n\n\ndef orientation_compatible(species: str, o1: str, o2: str, allow_opposite_lynx: bool = False) -> tuple[bool, str]:\n    o1 = (o1 or \"unknown\").lower()\n    o2 = (o2 or \"unknown\").lower()\n    if species != \"LynxID2025\":\n        return True, \"not_applicable\"\n    side_set = {\"left\", \"right\"}\n    if o1 in side_set and o2 in side_set and o1 != o2:\n        return allow_opposite_lynx, \"lynx_opposite_flank\"\n    if o1 in {\"front\", \"back\"} or o2 in {\"front\", \"back\"}:\n        return True, \"lynx_weak_pose\"\n    return True, \"lynx_same_or_unknown\"\n\n\ndef shortlist_pairs(\n    items: list[PatternItem],\n    current_labels: dict[int, str],\n    alt_votes: dict[tuple[int, int], int],\n    config: SpeciesConfig,\n    top_k_scale: float,\n    pair_budget: int,\n) -> set[tuple[int, int]]:\n    image_ids = [it.image_id for it in items]\n    item_by_id = {it.image_id: it for it in items}\n    pairs = set(current_cluster_pairs(current_labels, config.max_cluster_pair_size))\n    pairs.update(alt_votes.keys())\n\n    vectors = np.stack([it.vector for it in items]).astype(np.float32)\n    sim = vectors @ vectors.T\n    np.fill_diagonal(sim, -np.inf)\n    top_k = max(1, int(round(config.top_k * top_k_scale)))\n    for i, image_id in enumerate(image_ids):\n        k = min(top_k, len(image_ids) - 1)\n        if k <= 0:\n            continue\n        idx = np.argpartition(-sim[i], kth=k - 1)[:k]\n        for j in idx:\n            other_id = image_ids[int(j)]\n            if other_id == image_id:\n                continue\n            a, b = pair_key(image_id, other_id)\n            ok, _ = orientation_compatible(\n                config.species,\n                item_by_id[a].orientation,\n                item_by_id[b].orientation,\n                allow_opposite_lynx=False,\n            )\n            if ok or current_labels.get(a) == current_labels.get(b):\n                pairs.add((a, b))\n\n    if len(pairs) <= pair_budget:\n        return pairs\n\n    id_to_idx = {image_id: idx for idx, image_id in enumerate(image_ids)}\n    priority = []\n    for a, b in pairs:\n        ia = id_to_idx[a]\n        ib = id_to_idx[b]\n        same_current = current_labels.get(a) == current_labels.get(b)\n        vote = alt_votes.get((a, b), 0)\n        priority.append((same_current, vote, float(sim[ia, ib]), a, b))\n    priority.sort(reverse=True)\n    return {pair_key(a, b) for _, _, _, a, b in priority[:pair_budget]}\n\n\ndef bf_matcher(detector_name: str):\n    if detector_name == \"sift\":\n        return cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)\n    return cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)\n\n\ndef local_match_score(\n    a: PatternItem,\n    b: PatternItem,\n    config: SpeciesConfig,\n    detector_name: str,\n    ratio_test: float,\n) -> dict:\n    visual_sim, part_score, part_cells, part_overlap = occlusion_aware_similarity(a, b)\n    if a.descriptors is None or b.descriptors is None or len(a.descriptors) < 4 or len(b.descriptors) < 4:\n        return {\n            \"inliers\": 0,\n            \"good_matches\": 0,\n            \"inlier_ratio\": 0.0,\n            \"spatial_coverage\": 0.0,\n            \"local_score\": max(0.0, visual_sim) * 0.12,\n            \"visual_sim\": visual_sim,\n            \"part_score\": part_score,\n            \"part_overlap\": part_overlap,\n            \"part_cells\": part_cells,\n            \"coverage_a\": float(a.visibility_coverage),\n            \"coverage_b\": float(b.visibility_coverage),\n        }\n    matcher = bf_matcher(detector_name)\n    try:\n        knn = matcher.knnMatch(a.descriptors, b.descriptors, k=2)\n    except Exception:\n        return {\n            \"inliers\": 0,\n            \"good_matches\": 0,\n            \"inlier_ratio\": 0.0,\n            \"spatial_coverage\": 0.0,\n            \"local_score\": max(0.0, visual_sim) * 0.12,\n            \"visual_sim\": visual_sim,\n            \"part_score\": part_score,\n            \"part_overlap\": part_overlap,\n            \"part_cells\": part_cells,\n            \"coverage_a\": float(a.visibility_coverage),\n            \"coverage_b\": float(b.visibility_coverage),\n        }\n    good = []\n    for pair in knn:\n        if len(pair) < 2:\n            continue\n        m, n = pair\n        if m.distance < ratio_test * n.distance:\n            good.append(m)\n    if len(good) < 4:\n        return {\n            \"inliers\": 0,\n            \"good_matches\": len(good),\n            \"inlier_ratio\": 0.0,\n            \"spatial_coverage\": 0.0,\n            \"local_score\": max(0.0, visual_sim) * 0.12,\n            \"visual_sim\": visual_sim,\n            \"part_score\": part_score,\n            \"part_overlap\": part_overlap,\n            \"part_cells\": part_cells,\n            \"coverage_a\": float(a.visibility_coverage),\n            \"coverage_b\": float(b.visibility_coverage),\n        }\n    pts_a = np.float32([a.keypoints[m.queryIdx] for m in good]).reshape(-1, 2)\n    pts_b = np.float32([b.keypoints[m.trainIdx] for m in good]).reshape(-1, 2)\n    _, inlier_mask = cv2.estimateAffinePartial2D(\n        pts_a,\n        pts_b,\n        method=cv2.RANSAC,\n        ransacReprojThreshold=config.ransac_reproj,\n        maxIters=2000,\n        confidence=0.995,\n    )\n    if inlier_mask is None:\n        inliers = 0\n        inlier_flags = np.zeros(len(good), dtype=bool)\n    else:\n        inlier_flags = inlier_mask.reshape(-1).astype(bool)\n        inliers = int(inlier_flags.sum())\n    denom = max(1, min(len(a.keypoints), len(b.keypoints), len(good)))\n    inlier_ratio = float(inliers / denom)\n    if inliers > 1:\n        pa = pts_a[inlier_flags]\n        pb = pts_b[inlier_flags]\n        cov_a = (float(pa[:, 0].max() - pa[:, 0].min()) * float(pa[:, 1].max() - pa[:, 1].min())) / max(\n            1.0, float(config.target_size[0] * config.target_size[1])\n        )\n        cov_b = (float(pb[:, 0].max() - pb[:, 0].min()) * float(pb[:, 1].max() - pb[:, 1].min())) / max(\n            1.0, float(config.target_size[0] * config.target_size[1])\n        )\n        spatial_coverage = float(min(1.0, math.sqrt(max(0.0, cov_a * cov_b)) * 4.0))\n    else:\n        spatial_coverage = 0.0\n    inlier_term = min(1.0, inliers / max(1.0, config.conservative.min_inliers * 1.4))\n    ratio_term = min(1.0, inlier_ratio / max(0.01, config.conservative.min_inlier_ratio * 1.2))\n    part_term = max(0.0, min(1.0, (part_score + 0.05) / 1.05))\n    overlap_term = min(1.0, part_overlap * 2.0)\n    local_score = (\n        0.42 * inlier_term\n        + 0.24 * ratio_term\n        + 0.10 * spatial_coverage\n        + 0.16 * part_term\n        + 0.04 * overlap_term\n        + 0.04 * max(0.0, min(1.0, (visual_sim + 0.1) / 1.1))\n    )\n    if part_cells < 2 and inliers < max(10, config.balanced.min_inliers):\n        local_score *= 0.68\n    if min(a.visibility_coverage, b.visibility_coverage) < 0.12 and inliers < max(12, config.balanced.min_inliers):\n        local_score *= 0.72\n    local_score *= min(1.0, 0.65 + 0.35 * min(a.quality, b.quality))\n    return {\n        \"inliers\": inliers,\n        \"good_matches\": len(good),\n        \"inlier_ratio\": inlier_ratio,\n        \"spatial_coverage\": spatial_coverage,\n        \"local_score\": float(local_score),\n        \"visual_sim\": visual_sim,\n        \"part_score\": part_score,\n        \"part_overlap\": part_overlap,\n        \"part_cells\": part_cells,\n        \"coverage_a\": float(a.visibility_coverage),\n        \"coverage_b\": float(b.visibility_coverage),\n    }\n\n\ndef accept_edge(row: dict, rule: EdgeRule) -> bool:\n    if row[\"species\"] == \"LynxID2025\" and row[\"orientation_rule\"] == \"lynx_opposite_flank\" and not rule.allow_lynx_opposite_side:\n        return False\n    alt_vote = int(row.get(\"alt_votes\", 0))\n    min_inliers = max(4, rule.min_inliers - min(6, alt_vote * 2))\n    min_ratio = max(0.05, rule.min_inlier_ratio - min(0.04, alt_vote * 0.012))\n    min_score = max(0.10, rule.min_score - min(0.06, alt_vote * 0.018))\n    # Occlusion-aware guard: missing SAM regions are allowed, but a merge\n    # still needs either several mutually visible body zones or very strong\n    # geometric evidence. This avoids treating a tiny unmasked fragment as a\n    # full fingerprint match.\n    if int(row.get(\"part_cells\", 0)) < 2 and int(row[\"inliers\"]) < (min_inliers + 4) and alt_vote == 0:\n        return False\n    if float(row.get(\"part_overlap\", 0.0)) < 0.08 and int(row[\"inliers\"]) < int(math.ceil(min_inliers * 1.5)) and alt_vote == 0:\n        return False\n    return (\n        int(row[\"inliers\"]) >= min_inliers\n        and float(row[\"inlier_ratio\"]) >= min_ratio\n        and float(row[\"local_score\"]) >= min_score\n    )\n\n\ndef evaluate_pairs_for_species(\n    species: str,\n    species_rows: pd.DataFrame,\n    source_submissions: dict[str, pd.DataFrame],\n    args: argparse.Namespace,\n    reports_dir: Path,\n) -> tuple[list[PatternItem], pd.DataFrame, dict]:\n    config = SPECIES_CONFIGS[species]\n    detector_name, detector = create_detector(config.max_keypoints)\n    rows = species_rows.sort_values(\"image_id\").to_dict(\"records\")\n    if args.max_images_per_species:\n        rows = rows[: args.max_images_per_species]\n    if args.smoke:\n        rows = rows[: min(len(rows), 42)]\n\n    if args.save_roi_preview:\n        save_roi_preview(rows, reports_dir / f\"{VERSION}_{species}_roi_preview.jpg\", config, args.max_side)\n\n    items: list[PatternItem] = []\n    failures = 0\n    for idx, row in enumerate(rows, start=1):\n        try:\n            items.append(extract_pattern_item(row, config, args.max_side, detector_name, detector))\n        except Exception as exc:\n            failures += 1\n            fallback_vec = np.zeros(1024 + 56, dtype=np.float32)\n            n_parts = part_grid(config)[0] * part_grid(config)[1]\n            fallback_parts = np.zeros((n_parts, 96), dtype=np.float32)\n            fallback_vis = np.zeros(n_parts, dtype=bool)\n            items.append(\n                PatternItem(\n                    image_id=int(row[\"image_id\"]),\n                    row_idx=int(row.get(\"row_idx\", row[\"image_id\"])),\n                    species=species,\n                    orientation=str(row.get(\"orientation\", \"unknown\")).lower(),\n                    view_path=str(row.get(\"view_path\", row.get(\"source_path\", \"\"))),\n                    view_source=\"failed\",\n                    quality=0.0,\n                    keypoints=np.zeros((0, 2), dtype=np.float32),\n                    descriptors=None,\n                    vector=fallback_vec,\n                    fallback_vector=fallback_vec,\n                    part_vectors=fallback_parts,\n                    part_visibility=fallback_vis,\n                    fallback_part_vectors=fallback_parts,\n                    fallback_part_visibility=fallback_vis,\n                    visibility_coverage=0.0,\n                    n_keypoints=0,\n                )\n            )\n            print(f\"[warn] {species}: feature extraction failed for image_id={row.get('image_id')}: {exc}\")\n        if idx % 100 == 0:\n            print(f\"[{species}] extracted {idx}/{len(rows)}\")\n\n    current_labels = labels_for_species(source_submissions[\"current_best\"], species_rows)\n    if args.smoke or args.max_images_per_species:\n        keep_ids = {it.image_id for it in items}\n        current_labels = {k: v for k, v in current_labels.items() if k in keep_ids}\n    alt_votes = cluster_pair_votes(source_submissions, species_rows[species_rows[\"image_id\"].isin(current_labels)], config)\n    alt_votes = {k: v for k, v in alt_votes.items() if k[0] in current_labels and k[1] in current_labels}\n    pairs = shortlist_pairs(items, current_labels, alt_votes, config, args.top_k_scale, args.pair_budget_per_species)\n    item_by_id = {it.image_id: it for it in items}\n\n    score_rows = []\n    for idx, (a_id, b_id) in enumerate(sorted(pairs), start=1):\n        a = item_by_id.get(a_id)\n        b = item_by_id.get(b_id)\n        if a is None or b is None:\n            continue\n        orient_ok, orient_rule = orientation_compatible(species, a.orientation, b.orientation, allow_opposite_lynx=True)\n        scores = local_match_score(a, b, config, detector_name, config.ratio_test)\n        if species == \"LynxID2025\" and orient_rule == \"lynx_opposite_flank\":\n            scores[\"local_score\"] *= 0.55\n        same_current = current_labels.get(a_id) == current_labels.get(b_id)\n        row = {\n            \"species\": species,\n            \"image_id_a\": a_id,\n            \"image_id_b\": b_id,\n            \"orientation_a\": a.orientation,\n            \"orientation_b\": b.orientation,\n            \"orientation_rule\": orient_rule,\n            \"same_current_cluster\": bool(same_current),\n            \"alt_votes\": int(alt_votes.get((a_id, b_id), 0)),\n            \"detector\": detector_name,\n            \"kp_a\": int(a.n_keypoints),\n            \"kp_b\": int(b.n_keypoints),\n            **scores,\n        }\n        row[\"accept_conservative\"] = accept_edge(row, config.conservative)\n        row[\"accept_balanced\"] = accept_edge(row, config.balanced)\n        row[\"accept_swing\"] = accept_edge(row, config.swing)\n        score_rows.append(row)\n        if idx % 5000 == 0:\n            print(f\"[{species}] scored {idx}/{len(pairs)} pairs\")\n\n    pair_scores = pd.DataFrame(score_rows)\n    info = {\n        \"species\": species,\n        \"n_images\": len(items),\n        \"feature_failures\": failures,\n        \"detector\": detector_name,\n        \"n_pairs_scored\": int(len(pair_scores)),\n        \"mean_keypoints\": float(np.mean([it.n_keypoints for it in items])) if items else 0.0,\n        \"median_keypoints\": float(np.median([it.n_keypoints for it in items])) if items else 0.0,\n        \"mean_visibility_coverage\": float(np.mean([it.visibility_coverage for it in items])) if items else 0.0,\n        \"median_visibility_coverage\": float(np.median([it.visibility_coverage for it in items])) if items else 0.0,\n        \"resolved_sam_or_mask_views\": int(sum(it.view_source != \"original\" for it in items)),\n    }\n    if not pair_scores.empty:\n        for col in [\"accept_conservative\", \"accept_balanced\", \"accept_swing\"]:\n            info[col] = int(pair_scores[col].sum())\n        info[\"max_inliers\"] = int(pair_scores[\"inliers\"].max())\n        info[\"max_local_score\"] = float(pair_scores[\"local_score\"].max())\n        info[\"median_part_overlap\"] = float(pair_scores[\"part_overlap\"].median())\n        info[\"max_part_score\"] = float(pair_scores[\"part_score\"].max())\n    return items, pair_scores, info\n\n\ndef summarize_submission(submission: pd.DataFrame, test_rows: pd.DataFrame, current: pd.DataFrame, variant: str) -> list[dict]:\n    current_map = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n    sub_map = dict(zip(submission[\"image_id\"].astype(int), submission[\"cluster\"].astype(str)))\n    rows = []\n    for species in SPECIES_ORDER:\n        ids = test_rows.loc[test_rows[\"species_id\"].eq(species), \"image_id\"].astype(int).tolist()\n        labels = [sub_map[i] for i in ids]\n        counts = pd.Series(labels).value_counts()\n        changed = sum(1 for i in ids if current_map.get(i) != sub_map.get(i))\n        rows.append(\n            {\n                \"variant\": variant,\n                \"species\": species,\n                \"n_images\": len(ids),\n                \"n_clusters\": int(counts.shape[0]),\n                \"max_cluster_size\": int(counts.max()) if not counts.empty else 0,\n                \"singletons\": int((counts == 1).sum()) if not counts.empty else 0,\n                \"rows_changed_vs_current\": int(changed),\n            }\n        )\n    return rows\n\n\ndef relabel_components(image_to_component: dict[int, object], species: str, variant: str) -> dict[int, str]:\n    component_order: dict[object, int] = {}\n    labels: dict[int, str] = {}\n    for image_id in sorted(image_to_component):\n        comp = image_to_component[image_id]\n        if comp not in component_order:\n            component_order[comp] = len(component_order)\n        labels[image_id] = f\"cluster_{species}_{variant}_{component_order[comp]}\"\n    return labels\n\n\ndef merge_variant_for_species(\n    species: str,\n    species_ids: list[int],\n    current_labels: dict[int, str],\n    pair_scores: pd.DataFrame,\n    profile: str,\n) -> dict[int, str]:\n    config = SPECIES_CONFIGS[species]\n    clusters = sorted({current_labels[i] for i in species_ids})\n    uf = UnionFind(clusters)\n    if pair_scores.empty:\n        return {i: current_labels[i] for i in species_ids}\n    accept_col = f\"accept_{profile}\"\n    for row in pair_scores[pair_scores[accept_col]].itertuples(index=False):\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        ca = current_labels.get(a)\n        cb = current_labels.get(b)\n        if ca is None or cb is None:\n            continue\n        if ca != cb:\n            uf.union(ca, cb)\n    return {i: str(uf.find(current_labels[i])) for i in species_ids}\n\n\ndef splitmerge_variant_for_species(\n    species: str,\n    species_ids: list[int],\n    current_labels: dict[int, str],\n    pair_scores: pd.DataFrame,\n) -> dict[int, str]:\n    config = SPECIES_CONFIGS[species]\n    if species not in {\"LynxID2025\", \"TexasHornedLizards\"}:\n        return merge_variant_for_species(species, species_ids, current_labels, pair_scores, \"swing\")\n\n    uf = UnionFind(species_ids)\n    groups: dict[str, list[int]] = {}\n    for image_id in species_ids:\n        groups.setdefault(current_labels[image_id], []).append(image_id)\n\n    for members in groups.values():\n        if len(members) <= config.preserve_clusters_up_to:\n            anchor = members[0]\n            for other in members[1:]:\n                uf.union(anchor, other)\n\n    if not pair_scores.empty:\n        accepted = pair_scores[pair_scores[\"accept_swing\"]].copy()\n        for row in accepted.itertuples(index=False):\n            a = int(row.image_id_a)\n            b = int(row.image_id_b)\n            if a not in current_labels or b not in current_labels:\n                continue\n            ca = current_labels[a]\n            cb = current_labels[b]\n            same_current = ca == cb\n            if same_current or bool(row.accept_balanced) or int(row.alt_votes) > 0:\n                uf.union(a, b)\n\n    comp_by_id = {i: uf.find(i) for i in species_ids}\n    return relabel_components(comp_by_id, species, \"v7_swing_splitmerge\")\n\n\ndef build_submission_variants(\n    test_rows: pd.DataFrame,\n    source_submissions: dict[str, pd.DataFrame],\n    pair_scores_by_species: dict[str, pd.DataFrame],\n) -> dict[str, pd.DataFrame]:\n    current = source_submissions[\"current_best\"].copy()\n    variants: dict[str, pd.DataFrame] = {}\n    current_base = current.copy()\n    variants[\"current_best_passthrough\"] = current_base\n\n    species_rows_by_name = {\n        species: test_rows[test_rows[\"species_id\"].eq(species)].sort_values(\"image_id\").copy()\n        for species in SPECIES_ORDER\n    }\n    current_labels_by_species = {\n        species: labels_for_species(current, rows) for species, rows in species_rows_by_name.items()\n    }\n\n    for profile in [\"conservative\", \"balanced\", \"swing\"]:\n        sub = current.copy()\n        label_updates: dict[int, str] = {}\n        for species, rows in species_rows_by_name.items():\n            ids = rows[\"image_id\"].astype(int).tolist()\n            pair_scores = pair_scores_by_species.get(species, pd.DataFrame())\n            # SeaTurtle is already the strongest local branch; keep it guarded in all but swing.\n            effective_profile = \"conservative\" if species == \"SeaTurtleID2022\" and profile != \"swing\" else profile\n            labels = merge_variant_for_species(\n                species,\n                ids,\n                current_labels_by_species[species],\n                pair_scores,\n                effective_profile,\n            )\n            label_updates.update(labels)\n        original_map = dict(zip(sub[\"image_id\"].astype(int), sub[\"cluster\"].astype(str)))\n        sub[\"cluster\"] = sub[\"image_id\"].astype(int).map(lambda i: label_updates.get(i, original_map[i]))\n        variants[f\"current_plus_{profile}_pattern_merges\"] = sub\n\n    split_sub = current.copy()\n    split_updates: dict[int, str] = {}\n    for species, rows in species_rows_by_name.items():\n        ids = rows[\"image_id\"].astype(int).tolist()\n        labels = splitmerge_variant_for_species(\n            species,\n            ids,\n            current_labels_by_species[species],\n            pair_scores_by_species.get(species, pd.DataFrame()),\n        )\n        split_updates.update(labels)\n    split_original_map = dict(zip(split_sub[\"image_id\"].astype(int), split_sub[\"cluster\"].astype(str)))\n    split_sub[\"cluster\"] = split_sub[\"image_id\"].astype(int).map(lambda i: split_updates.get(i, split_original_map[i]))\n    variants[\"swing_split_large_lynx_texas\"] = split_sub\n    return variants\n\n\ndef update_experiment_log(output_root: Path, summary: dict, candidate_report: pd.DataFrame, best_path: Path) -> None:\n    log_path = Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\current_wildfusion_graph_v20260423\\experiment_log.json\")\n    if not log_path.exists():\n        return\n    try:\n        log = json.loads(log_path.read_text(encoding=\"utf-8\"))\n    except Exception:\n        return\n    entry = {\n        \"run_id\": VERSION,\n        \"date\": \"2026-04-26\",\n        \"status\": \"implemented_smoke_tested_ready_for_kaggle\",\n        \"goal\": \"Final-swing species-specific local fingerprint ensemble over current 0.29758 hybrid.\",\n        \"output_root\": str(output_root),\n        \"recommended_first_submission\": str(best_path),\n        \"summary\": summary,\n        \"candidate_report_preview\": candidate_report.to_dict(\"records\")[:20],\n    }\n    if isinstance(log, list):\n        log = [run for run in log if not (isinstance(run, dict) and run.get(\"run_id\") == VERSION)]\n        log.append(entry)\n    elif isinstance(log, dict):\n        runs = log.setdefault(\"runs\", [])\n        log[\"runs\"] = [run for run in runs if not (isinstance(run, dict) and run.get(\"run_id\") == VERSION)]\n        log[\"runs\"].append(entry)\n    else:\n        return\n    log_path.write_text(json.dumps(log, indent=2), encoding=\"utf-8\")\n\n\ndef main() -> None:\n    random.seed(SEED)\n    np.random.seed(SEED)\n    args = parse_args()\n    if args.smoke:\n        args.max_images_per_species = args.max_images_per_species or 42\n        args.pair_budget_per_species = min(args.pair_budget_per_species, 2500)\n        args.top_k_scale = min(args.top_k_scale, 0.55)\n\n    data_root = find_data_root(args.data_root)\n    sam_manifest = find_sam_manifest(args.sam_manifest)\n    output_root = Path(args.output_root) if args.output_root else Path.cwd() / f\"animalclef_{VERSION}\"\n    reports_dir = output_root / \"reports\"\n    submissions_dir = output_root / \"submissions\"\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    submissions_dir.mkdir(parents=True, exist_ok=True)\n\n    test_rows, manifest_info = prepare_test_metadata(data_root, sam_manifest)\n    selected_species = [s for s in args.species if s in SPECIES_CONFIGS]\n    test_rows = test_rows[test_rows[\"species_id\"].isin(SPECIES_ORDER)].copy()\n    source_paths = find_submission_sources(args, data_root)\n    source_submissions = {name: load_submission(path) for name, path in source_paths.items()}\n\n    print(f\"VERSION={VERSION}\")\n    print(f\"data_root={data_root}\")\n    print(f\"sam_manifest={sam_manifest}\")\n    print(f\"output_root={output_root}\")\n    print(\"submission sources:\")\n    for name, path in source_paths.items():\n        print(f\"  {name}: {path}\")\n\n    pair_scores_by_species: dict[str, pd.DataFrame] = {}\n    item_infos: list[dict] = []\n    for species in selected_species:\n        species_rows = test_rows[test_rows[\"species_id\"].eq(species)].copy()\n        items, pair_scores, info = evaluate_pairs_for_species(species, species_rows, source_submissions, args, reports_dir)\n        pair_scores_by_species[species] = pair_scores\n        item_infos.append(info)\n        item_manifest = pd.DataFrame(\n            [\n                {\n                    \"species\": it.species,\n                    \"image_id\": it.image_id,\n                    \"row_idx\": it.row_idx,\n                    \"orientation\": it.orientation,\n                    \"view_path\": it.view_path,\n                    \"view_source\": it.view_source,\n                    \"quality\": it.quality,\n                    \"visibility_coverage\": it.visibility_coverage,\n                    \"visible_parts\": int(it.part_visibility.sum()),\n                    \"fallback_visible_parts\": int(it.fallback_part_visibility.sum()),\n                    \"n_keypoints\": it.n_keypoints,\n                }\n                for it in items\n            ]\n        )\n        item_manifest.to_csv(reports_dir / f\"{VERSION}_{species}_item_manifest.csv\", index=False)\n        pair_scores.to_csv(reports_dir / f\"{VERSION}_{species}_pair_scores.csv\", index=False)\n\n    nonempty_pair_scores = [df for df in pair_scores_by_species.values() if not df.empty]\n    all_pair_scores = pd.concat(nonempty_pair_scores, ignore_index=True) if nonempty_pair_scores else pd.DataFrame()\n    all_pair_scores.to_csv(reports_dir / f\"{VERSION}_all_pair_scores.csv\", index=False)\n\n    variants = build_submission_variants(test_rows, source_submissions, pair_scores_by_species)\n    current = source_submissions[\"current_best\"]\n    report_rows = []\n    written_paths = {}\n    for variant, sub in variants.items():\n        out_path = submissions_dir / f\"submission_{VERSION}_{variant}.csv\"\n        sub.to_csv(out_path, index=False)\n        written_paths[variant] = str(out_path)\n        report_rows.extend(summarize_submission(sub, test_rows, current, variant))\n\n    candidate_report = pd.DataFrame(report_rows)\n    candidate_report.to_csv(reports_dir / f\"{VERSION}_candidate_report.csv\", index=False)\n\n    recommended = submissions_dir / f\"submission_{VERSION}_current_plus_conservative_pattern_merges.csv\"\n    summary = {\n        \"version\": VERSION,\n        \"data_root\": str(data_root),\n        \"sam_manifest\": str(sam_manifest) if sam_manifest else None,\n        \"manifest_info\": manifest_info,\n        \"selected_species\": selected_species,\n        \"source_paths\": {k: str(v) for k, v in source_paths.items()},\n        \"item_infos\": item_infos,\n        \"written_submissions\": written_paths,\n        \"recommended_first_submission\": str(recommended),\n        \"notes\": [\n            \"Conservative/balanced/swing merge variants preserve the current 0.29758 hybrid and only merge clusters with local pattern evidence.\",\n            \"Occlusion-aware visibility grids ignore SAM-removed or hand-corrupted body zones instead of treating missing parts as mismatches.\",\n            \"swing_split_large_lynx_texas is the high-risk/high-upside file; it can repair oversized Lynx/Texas clusters but may over-split.\",\n            \"SeaTurtle uses guarded rules because p06 is already very strong locally.\",\n        ],\n    }\n    (reports_dir / f\"{VERSION}_summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    update_experiment_log(output_root, summary, candidate_report, recommended)\n\n    print(\"candidate report:\")\n    print(candidate_report.to_string(index=False))\n    print(f\"wrote {recommended}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
Path("species_anatomy_template_registration_v20260426.py").write_text("from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport math\nimport random\nimport sys\nfrom collections import deque\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFile\n\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nTHIS_DIR = Path(__file__).resolve().parent\nif str(THIS_DIR) not in sys.path:\n    sys.path.insert(0, str(THIS_DIR))\n\nimport species_fingerprint_final_swing_v20260426 as core\n\n\nVERSION = \"species_anatomy_template_registration_v20260426\"\nSEED = 20260426\nBACKGROUND = np.array([238, 238, 232], dtype=np.uint8)\n\n\n@dataclass(frozen=True)\nclass AnatomyConfig:\n    species: str\n    target_size: tuple[int, int]\n    template_kind: str\n    top_k: int\n    grid: tuple[int, int]\n    strict_thr: float\n    balanced_thr: float\n    aggressive_thr: float\n    force_thr: float\n    min_overlap: float\n    shift_px: tuple[int, ...]\n\n\nCONFIGS: dict[str, AnatomyConfig] = {\n    \"LynxID2025\": AnatomyConfig(\n        species=\"LynxID2025\",\n        target_size=(288, 168),\n        template_kind=\"lynx_flank_spot_template\",\n        top_k=58,\n        grid=(6, 12),\n        strict_thr=0.54,\n        balanced_thr=0.49,\n        aggressive_thr=0.45,\n        force_thr=0.40,\n        min_overlap=0.11,\n        shift_px=(-10, 0, 10),\n    ),\n    \"SalamanderID2025\": AnatomyConfig(\n        species=\"SalamanderID2025\",\n        target_size=(192, 64),\n        template_kind=\"salamander_centerline_unwrap\",\n        top_k=65,\n        grid=(4, 18),\n        strict_thr=0.56,\n        balanced_thr=0.50,\n        aggressive_thr=0.46,\n        force_thr=0.42,\n        min_overlap=0.14,\n        shift_px=(-8, 0, 8),\n    ),\n    \"SeaTurtleID2022\": AnatomyConfig(\n        species=\"SeaTurtleID2022\",\n        target_size=(224, 224),\n        template_kind=\"turtle_head_scute_template\",\n        top_k=40,\n        grid=(8, 8),\n        strict_thr=0.58,\n        balanced_thr=0.52,\n        aggressive_thr=0.47,\n        force_thr=0.42,\n        min_overlap=0.14,\n        shift_px=(-10, 0, 10),\n    ),\n    \"TexasHornedLizards\": AnatomyConfig(\n        species=\"TexasHornedLizards\",\n        target_size=(224, 320),\n        template_kind=\"texas_belly_dot_template\",\n        top_k=75,\n        grid=(10, 8),\n        strict_thr=0.50,\n        balanced_thr=0.46,\n        aggressive_thr=0.42,\n        force_thr=0.38,\n        min_overlap=0.12,\n        shift_px=(-12, 0, 12),\n    ),\n}\n\n\n@dataclass\nclass AnatomyItem:\n    image_id: int\n    row_idx: int\n    identity: str\n    split: str\n    species: str\n    orientation: str\n    source_path: str\n    sam_view_path: str\n    view_path: str\n    mask_path: str\n    view_source: str\n    quality: float\n    template_rgb: np.ndarray\n    template_mask: np.ndarray\n    pattern: np.ndarray\n    vector: np.ndarray\n    points: np.ndarray\n    debug: dict\n\n\nclass UnionFind:\n    def __init__(self, values: Iterable[int]):\n        self.parent = {int(v): int(v) for v in values}\n        self.size = {int(v): 1 for v in values}\n\n    def find(self, value: int) -> int:\n        value = int(value)\n        if value not in self.parent:\n            self.parent[value] = value\n            self.size[value] = 1\n            return value\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a: int, b: int) -> int:\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return ra\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n        return ra\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            \"AnimalCLEF2026 anatomy-template registration branch. It is not a \"\n            \"current-best safeguard merge. It uses SAM-clean foreground views, \"\n            \"species-specific anatomy canonicalization, occlusion-aware template \"\n            \"overlap, train-label validation where available, and independent \"\n            \"test clustering.\"\n        )\n    )\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--sam-manifest\", type=str, default=None)\n    parser.add_argument(\"--output-root\", type=str, default=None)\n    parser.add_argument(\"--species\", nargs=\"*\", default=core.SPECIES_ORDER)\n    parser.add_argument(\"--profiles\", nargs=\"*\", default=[\"strict\", \"balanced\", \"aggressive\", \"force\"])\n    parser.add_argument(\"--max-side\", type=int, default=760)\n    parser.add_argument(\"--top-k-scale\", type=float, default=1.0)\n    parser.add_argument(\"--test-pair-budget-per-species\", type=int, default=90000)\n    parser.add_argument(\"--train-pair-budget-per-species\", type=int, default=5000)\n    parser.add_argument(\"--max-train-images-per-species\", type=int, default=700)\n    parser.add_argument(\"--max-test-images-per-species\", type=int, default=0)\n    parser.add_argument(\"--max-train-per-identity\", type=int, default=8)\n    parser.add_argument(\"--skip-train-validation\", action=\"store_true\")\n    parser.add_argument(\"--save-visualizations\", action=\"store_true\")\n    parser.add_argument(\"--visual-limit\", type=int, default=14)\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef find_data_root(user_value: str | None) -> Path:\n    candidates: list[Path] = []\n    if user_value:\n        candidates.append(Path(user_value))\n    candidates.extend(\n        [\n            Path(\"/kaggle/input/animal-clef-2026\"),\n            Path(\"/kaggle/input/competitions/animal-clef-2026\"),\n            Path.cwd() / \"animal-clef-2026\",\n            Path.cwd().parent / \"animal-clef-2026\",\n            Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\animal-clef-2026\"),\n        ]\n    )\n    for root in candidates:\n        if (root / \"metadata.csv\").exists() and (root / \"sample_submission.csv\").exists():\n            return root.resolve()\n    raise FileNotFoundError(\"Could not locate AnimalCLEF2026 data root.\")\n\n\ndef find_sam_manifest(user_value: str | None) -> Path | None:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    direct = [\n        Path(\"/kaggle/working/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/reports/view_manifest_sam3_all_species.csv\"),\n        Path.cwd() / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n        Path.cwd().parent / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n    ]\n    for p in direct:\n        if p.exists():\n            return p.resolve()\n    for base in [Path(\"/kaggle/input\"), Path.cwd(), Path.cwd().parent]:\n        if not base.exists():\n            continue\n        try:\n            matches = list(base.rglob(\"view_manifest_sam3_all_species.csv\"))\n        except Exception:\n            matches = []\n        if matches:\n            matches.sort(key=lambda x: len(str(x)))\n            return matches[0].resolve()\n    return None\n\n\ndef export_root_from_manifest(manifest_path: Path) -> Path:\n    if manifest_path.parent.name == \"reports\":\n        return manifest_path.parent.parent\n    return manifest_path.parent\n\n\ndef remap_export_path(path_value: object, export_root: Path | None) -> Path | None:\n    if path_value is None or pd.isna(path_value):\n        return None\n    s = str(path_value).strip()\n    if not s:\n        return None\n    p = Path(s)\n    if p.exists():\n        return p.resolve()\n    if export_root is None:\n        return None\n    normalized = s.replace(\"\\\\\", \"/\")\n    markers = [\n        \"animalclef_sam3_views_cache/\",\n        \"views/\",\n        \"mask_loose_square/\",\n        \"mask_full_square/\",\n    ]\n    for marker in markers:\n        if marker not in normalized:\n            continue\n        rel = normalized.split(marker, 1)[1]\n        if marker != \"animalclef_sam3_views_cache/\":\n            rel = marker + rel\n        candidate = export_root / Path(rel)\n        if candidate.exists():\n            return candidate.resolve()\n    return None\n\n\ndef prepare_metadata(data_root: Path, sam_manifest: Path | None) -> tuple[pd.DataFrame, dict]:\n    metadata = pd.read_csv(data_root / \"metadata.csv\").reset_index(drop=True)\n    if \"row_idx\" not in metadata.columns:\n        metadata[\"row_idx\"] = np.arange(len(metadata), dtype=np.int64)\n    if \"species_id\" not in metadata.columns:\n        if \"dataset\" in metadata.columns:\n            metadata[\"species_id\"] = metadata[\"dataset\"].astype(str)\n        else:\n            metadata[\"species_id\"] = metadata[\"path\"].str.replace(\"\\\\\", \"/\", regex=False).str.split(\"/\").str[1]\n    if \"split\" not in metadata.columns:\n        metadata[\"split\"] = np.where(metadata[\"path\"].str.contains(\"/test/\"), \"test\", \"train\")\n    if \"orientation\" not in metadata.columns:\n        metadata[\"orientation\"] = \"unknown\"\n    if \"identity\" not in metadata.columns:\n        metadata[\"identity\"] = \"\"\n    metadata[\"source_path\"] = metadata[\"path\"].map(lambda p: str(data_root / str(p)))\n    metadata[\"view_path\"] = metadata[\"source_path\"].astype(str)\n    metadata[\"view_source\"] = \"original_fallback\"\n    metadata[\"sam_view_path\"] = \"\"\n    metadata[\"mask_path\"] = \"\"\n    metadata[\"mask_ok\"] = False\n\n    info = {\n        \"manifest_path\": str(sam_manifest) if sam_manifest else None,\n        \"manifest_rows\": 0,\n        \"resolved_sam_views\": 0,\n        \"resolved_masks\": 0,\n    }\n    if sam_manifest is None:\n        return metadata, info\n\n    manifest = pd.read_csv(sam_manifest)\n    info[\"manifest_rows\"] = int(len(manifest))\n    export_root = export_root_from_manifest(sam_manifest)\n    merge_key = \"row_idx\" if \"row_idx\" in manifest.columns else \"image_id\" if \"image_id\" in manifest.columns else None\n    if merge_key is None:\n        return metadata, info\n    merged = metadata.merge(manifest, on=merge_key, how=\"left\", suffixes=(\"\", \"_sam\"))\n\n    view_paths: list[str] = []\n    sam_view_paths: list[str] = []\n    mask_paths: list[str] = []\n    view_sources: list[str] = []\n    mask_ok: list[bool] = []\n    sam_count = 0\n    mask_count = 0\n    for row in merged.to_dict(\"records\"):\n        resolved_sam_view = None\n        for col in [\"loose_path\", \"mask_loose_path\", \"mask_full_path\", \"view_path\"]:\n            if col in row:\n                resolved_sam_view = remap_export_path(row.get(col), export_root)\n                if resolved_sam_view is not None:\n                    break\n        resolved_mask = None\n        for col in [\"mask_path\", \"binary_mask_path\"]:\n            if col in row:\n                resolved_mask = remap_export_path(row.get(col), export_root)\n                if resolved_mask is not None:\n                    break\n        if resolved_sam_view is not None:\n            view_paths.append(str(resolved_sam_view))\n            sam_view_paths.append(str(resolved_sam_view))\n            view_sources.append(\"sam_clean_primary\")\n            sam_count += 1\n        else:\n            view_paths.append(str(row[\"source_path\"]))\n            sam_view_paths.append(\"\")\n            view_sources.append(\"original_fallback\")\n        if resolved_mask is not None:\n            mask_paths.append(str(resolved_mask))\n            mask_ok.append(True)\n            mask_count += 1\n        else:\n            mask_paths.append(\"\")\n            mask_ok.append(False)\n    merged[\"view_path\"] = view_paths\n    merged[\"sam_view_path\"] = sam_view_paths\n    merged[\"view_source\"] = view_sources\n    merged[\"mask_path\"] = mask_paths\n    merged[\"mask_ok\"] = mask_ok\n    info[\"resolved_sam_views\"] = int(sam_count)\n    info[\"resolved_masks\"] = int(mask_count)\n    return merged, info\n\n\ndef normalize(vec: np.ndarray) -> np.ndarray:\n    vec = vec.astype(np.float32, copy=False)\n    norm = float(np.linalg.norm(vec))\n    if norm > 0:\n        vec = vec / norm\n    return vec.astype(np.float32, copy=False)\n\n\ndef fill_holes(mask: np.ndarray) -> np.ndarray:\n    m = np.where(mask > 0, 255, 0).astype(np.uint8)\n    h, w = m.shape[:2]\n    flood = m.copy()\n    ff_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)\n    cv2.floodFill(flood, ff_mask, (0, 0), 255)\n    holes = cv2.bitwise_not(flood)\n    return cv2.bitwise_or(m, holes)\n\n\ndef clean_mask(mask: np.ndarray, rgb_shape: tuple[int, int] | None = None) -> np.ndarray:\n    m = np.where(mask > 0, 255, 0).astype(np.uint8)\n    if rgb_shape is not None and m.shape[:2] != rgb_shape:\n        m = cv2.resize(m, (rgb_shape[1], rgb_shape[0]), interpolation=cv2.INTER_NEAREST)\n        m = np.where(m > 0, 255, 0).astype(np.uint8)\n    if float(m.mean() / 255.0) < 0.01:\n        return np.full(m.shape[:2], 255, dtype=np.uint8)\n    h, w = m.shape[:2]\n    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))\n    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))\n    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k1)\n    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k2)\n    m = fill_holes(m)\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, 8)\n    if n <= 1:\n        return m\n    areas = stats[1:, cv2.CC_STAT_AREA]\n    largest = int(areas.max()) if len(areas) else 0\n    keep = np.zeros_like(m)\n    min_area = max(25, int(largest * 0.035), int(h * w * 0.0015))\n    for idx in range(1, n):\n        if int(stats[idx, cv2.CC_STAT_AREA]) >= min_area:\n            keep[labels == idx] = 255\n    if float(keep.mean() / 255.0) < 0.01:\n        biggest = 1 + int(np.argmax(areas))\n        keep[labels == biggest] = 255\n    return keep.astype(np.uint8)\n\n\ndef read_rgb_mask(row: dict, max_side: int) -> tuple[np.ndarray, np.ndarray, float]:\n    rgb, mask = core.read_rgb_with_optional_mask(row.get(\"view_path\", row.get(\"source_path\")), row.get(\"mask_path\"), max_side)\n    if mask is None:\n        mask = core.estimate_foreground_mask(rgb)\n        quality = 0.86\n    else:\n        quality = 1.0\n    mask = clean_mask(mask, rgb.shape[:2])\n    coverage = float(mask.mean() / 255.0)\n    if coverage < 0.015 or coverage > 0.98:\n        mask = core.estimate_foreground_mask(rgb)\n        mask = clean_mask(mask, rgb.shape[:2])\n        quality *= 0.78\n    return rgb, mask, quality\n\n\ndef crop_to_mask(rgb: np.ndarray, mask: np.ndarray, pad_frac: float = 0.06) -> tuple[np.ndarray, np.ndarray]:\n    x1, y1, x2, y2 = core.bbox_from_mask(mask, pad_frac)\n    return rgb[y1:y2, x1:x2].copy(), mask[y1:y2, x1:x2].copy()\n\n\ndef pca_align_template(\n    rgb: np.ndarray,\n    mask: np.ndarray,\n    target_size: tuple[int, int],\n    major_axis: str = \"horizontal\",\n    pad_frac: float = 0.06,\n) -> tuple[np.ndarray, np.ndarray, dict]:\n    crop_rgb, crop_mask = crop_to_mask(rgb, mask, pad_frac)\n    angle = core.pca_angle_degrees(crop_mask)\n    if major_axis == \"vertical\":\n        rotate_angle = 90.0 - angle\n    else:\n        rotate_angle = -angle\n    if abs(rotate_angle) > 1.5:\n        crop_rgb, crop_mask = core.rotate_bound(crop_rgb, crop_mask, rotate_angle)\n        crop_rgb, crop_mask = crop_to_mask(crop_rgb, crop_mask, 0.04)\n    w, h = target_size\n    out_rgb = cv2.resize(crop_rgb, (w, h), interpolation=cv2.INTER_AREA)\n    out_mask = cv2.resize(crop_mask, (w, h), interpolation=cv2.INTER_NEAREST)\n    out_mask = clean_mask(out_mask)\n    return out_rgb, out_mask, {\"pca_angle\": float(angle), \"rotate_angle\": float(rotate_angle)}\n\n\ndef skeletonize(mask: np.ndarray) -> np.ndarray:\n    img = np.where(mask > 0, 255, 0).astype(np.uint8)\n    skel = np.zeros_like(img)\n    element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))\n    for _ in range(700):\n        eroded = cv2.erode(img, element)\n        temp = cv2.dilate(eroded, element)\n        temp = cv2.subtract(img, temp)\n        skel = cv2.bitwise_or(skel, temp)\n        img = eroded\n        if cv2.countNonZero(img) == 0:\n            break\n    return skel\n\n\ndef bfs_farthest(start: int, neighbors: list[list[int]]) -> tuple[int, np.ndarray]:\n    n = len(neighbors)\n    parent = np.full(n, -1, dtype=np.int32)\n    dist = np.full(n, -1, dtype=np.int32)\n    q: deque[int] = deque([start])\n    dist[start] = 0\n    far = start\n    while q:\n        node = q.popleft()\n        if dist[node] > dist[far]:\n            far = node\n        for nxt in neighbors[node]:\n            if dist[nxt] >= 0:\n                continue\n            dist[nxt] = dist[node] + 1\n            parent[nxt] = node\n            q.append(nxt)\n    return far, parent\n\n\ndef longest_skeleton_path(skel: np.ndarray) -> np.ndarray | None:\n    ys, xs = np.where(skel > 0)\n    if len(xs) < 12:\n        return None\n    coords = np.stack([xs.astype(np.int32), ys.astype(np.int32)], axis=1)\n    index = -np.ones(skel.shape[:2], dtype=np.int32)\n    index[ys, xs] = np.arange(len(xs), dtype=np.int32)\n    neighbors: list[list[int]] = [[] for _ in range(len(xs))]\n    for idx, (x, y) in enumerate(coords):\n        for dy in (-1, 0, 1):\n            for dx in (-1, 0, 1):\n                if dx == 0 and dy == 0:\n                    continue\n                yy = y + dy\n                xx = x + dx\n                if 0 <= yy < skel.shape[0] and 0 <= xx < skel.shape[1]:\n                    j = int(index[yy, xx])\n                    if j >= 0:\n                        neighbors[idx].append(j)\n    degrees = np.array([len(n) for n in neighbors])\n    endpoints = np.where(degrees == 1)[0]\n    start = int(endpoints[0]) if len(endpoints) else 0\n    a, _ = bfs_farthest(start, neighbors)\n    b, parent = bfs_farthest(a, neighbors)\n    path_idx = [b]\n    cur = b\n    while cur != a and cur >= 0:\n        cur = int(parent[cur])\n        if cur >= 0:\n            path_idx.append(cur)\n        if len(path_idx) > len(coords):\n            break\n    if len(path_idx) < 8:\n        return None\n    path = coords[np.array(path_idx[::-1], dtype=np.int32)].astype(np.float32)\n    return path\n\n\ndef resample_polyline(path: np.ndarray, n_points: int) -> np.ndarray:\n    if path is None or len(path) < 2:\n        return np.zeros((n_points, 2), dtype=np.float32)\n    diffs = np.diff(path, axis=0)\n    seg = np.sqrt((diffs * diffs).sum(axis=1))\n    dist = np.concatenate([[0.0], np.cumsum(seg)])\n    total = float(dist[-1])\n    if total <= 1e-6:\n        return np.repeat(path[:1], n_points, axis=0).astype(np.float32)\n    target = np.linspace(0.0, total, n_points)\n    x = np.interp(target, dist, path[:, 0])\n    y = np.interp(target, dist, path[:, 1])\n    return np.stack([x, y], axis=1).astype(np.float32)\n\n\ndef unwrap_centerline_strip(\n    rgb: np.ndarray,\n    mask: np.ndarray,\n    target_size: tuple[int, int],\n) -> tuple[np.ndarray, np.ndarray, dict]:\n    crop_rgb, crop_mask = crop_to_mask(rgb, mask, 0.08)\n    h0, w0 = crop_mask.shape[:2]\n    scale = min(1.0, 380.0 / float(max(h0, w0)))\n    small_mask = cv2.resize(\n        crop_mask,\n        (max(1, int(round(w0 * scale))), max(1, int(round(h0 * scale)))),\n        interpolation=cv2.INTER_NEAREST,\n    )\n    small_mask = clean_mask(small_mask)\n    skel = skeletonize(small_mask)\n    path_small = longest_skeleton_path(skel)\n    if path_small is None:\n        aligned_rgb, aligned_mask, debug = pca_align_template(crop_rgb, crop_mask, target_size, \"horizontal\", 0.05)\n        debug[\"unwrap_fallback\"] = True\n        return aligned_rgb, aligned_mask, debug\n\n    path = path_small / max(scale, 1e-6)\n    strip_w, strip_h = target_size\n    path = resample_polyline(path, strip_w)\n    dt = cv2.distanceTransform(np.where(crop_mask > 0, 1, 0).astype(np.uint8), cv2.DIST_L2, 5)\n    hw = []\n    for x, y in path:\n        xi = int(np.clip(round(float(x)), 0, w0 - 1))\n        yi = int(np.clip(round(float(y)), 0, h0 - 1))\n        hw.append(float(dt[yi, xi]))\n    hw_arr = np.asarray(hw, dtype=np.float32)\n    valid = hw_arr[hw_arr > 1.0]\n    median_hw = float(np.median(valid)) if valid.size else max(4.0, min(h0, w0) * 0.08)\n    hw_arr = np.clip(hw_arr * 1.30, max(3.0, median_hw * 0.38), max(6.0, median_hw * 1.95))\n\n    tangent = np.gradient(path, axis=0)\n    norm = np.sqrt((tangent * tangent).sum(axis=1, keepdims=True))\n    tangent = tangent / np.maximum(norm, 1e-6)\n    normal = np.stack([-tangent[:, 1], tangent[:, 0]], axis=1)\n    offsets = np.linspace(-1.15, 1.15, strip_h, dtype=np.float32)\n    map_x = np.zeros((strip_h, strip_w), dtype=np.float32)\n    map_y = np.zeros((strip_h, strip_w), dtype=np.float32)\n    for col in range(strip_w):\n        p = path[col]\n        n = normal[col]\n        pts = p[None, :] + offsets[:, None] * hw_arr[col] * n[None, :]\n        map_x[:, col] = pts[:, 0]\n        map_y[:, col] = pts[:, 1]\n\n    strip_rgb = cv2.remap(\n        crop_rgb,\n        map_x,\n        map_y,\n        interpolation=cv2.INTER_LINEAR,\n        borderMode=cv2.BORDER_CONSTANT,\n        borderValue=(238, 238, 232),\n    )\n    strip_mask = cv2.remap(\n        crop_mask,\n        map_x,\n        map_y,\n        interpolation=cv2.INTER_NEAREST,\n        borderMode=cv2.BORDER_CONSTANT,\n        borderValue=0,\n    )\n    strip_mask = clean_mask(strip_mask)\n    debug = {\n        \"unwrap_fallback\": False,\n        \"path_points\": int(len(path_small)),\n        \"median_half_width\": float(median_hw),\n        \"small_scale\": float(scale),\n    }\n    return strip_rgb, strip_mask, debug\n\n\ndef clahe_u8(channel: np.ndarray) -> np.ndarray:\n    channel = np.clip(channel, 0, 255).astype(np.uint8)\n    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(channel)\n\n\ndef pattern_channels(rgb: np.ndarray, mask: np.ndarray, species: str) -> np.ndarray:\n    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    l = clahe_u8(lab[:, :, 0])\n    sat = clahe_u8(hsv[:, :, 1])\n    b = clahe_u8(lab[:, :, 2])\n    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)\n    edges = cv2.Canny(gray, 60, 160).astype(np.float32) / 255.0\n    if species == \"SalamanderID2025\":\n        yellow = cv2.addWeighted(b, 0.62, sat, 0.38, 0).astype(np.float32) / 255.0\n        dark = (255.0 - l.astype(np.float32)) / 255.0\n        pat = np.stack([yellow, dark, edges], axis=2)\n    elif species in {\"LynxID2025\", \"TexasHornedLizards\"}:\n        dark = (255.0 - l.astype(np.float32)) / 255.0\n        blackhat = cv2.morphologyEx(l, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)))\n        spot = cv2.addWeighted((blackhat.astype(np.float32) / 255.0), 0.55, dark, 0.45, 0)\n        texture = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)\n        texture = np.clip(np.abs(texture) / 95.0, 0, 1)\n        pat = np.stack([spot, dark, texture], axis=2)\n    else:\n        a = clahe_u8(lab[:, :, 1]).astype(np.float32) / 255.0\n        bb = b.astype(np.float32) / 255.0\n        ll = l.astype(np.float32) / 255.0\n        pat = np.stack([ll, a, 0.5 * bb + 0.5 * edges], axis=2)\n    valid = (mask > 0).astype(np.float32)\n    pat *= valid[:, :, None]\n    return pat.astype(np.float32)\n\n\ndef apply_species_template_mask(mask: np.ndarray, species: str) -> np.ndarray:\n    out = np.where(mask > 0, 255, 0).astype(np.uint8)\n    h, w = out.shape[:2]\n    if species == \"TexasHornedLizards\":\n        ellipse = np.zeros_like(out)\n        cv2.ellipse(\n            ellipse,\n            (w // 2, int(h * 0.54)),\n            (int(w * 0.40), int(h * 0.42)),\n            0,\n            0,\n            360,\n            255,\n            -1,\n        )\n        out = cv2.bitwise_and(out, ellipse)\n    elif species == \"LynxID2025\":\n        flank = np.zeros_like(out)\n        flank[int(h * 0.10) : int(h * 0.90), int(w * 0.06) : int(w * 0.96)] = 255\n        out = cv2.bitwise_and(out, flank)\n    elif species == \"SalamanderID2025\":\n        band = np.zeros_like(out)\n        band[int(h * 0.08) : int(h * 0.92), :] = 255\n        out = cv2.bitwise_and(out, band)\n    return out\n\n\ndef grid_descriptor(pattern: np.ndarray, mask: np.ndarray, grid: tuple[int, int]) -> np.ndarray:\n    gy, gx = grid\n    h, w, c = pattern.shape\n    feats: list[float] = []\n    valid = mask > 0\n    for iy in range(gy):\n        y1 = int(round(iy * h / gy))\n        y2 = int(round((iy + 1) * h / gy))\n        for ix in range(gx):\n            x1 = int(round(ix * w / gx))\n            x2 = int(round((ix + 1) * w / gx))\n            cell_mask = valid[y1:y2, x1:x2]\n            coverage = float(cell_mask.mean()) if cell_mask.size else 0.0\n            feats.append(coverage)\n            if coverage < 0.04:\n                feats.extend([0.0] * (c * 3))\n                continue\n            vals = pattern[y1:y2, x1:x2][cell_mask]\n            feats.extend(vals.mean(axis=0).astype(float).tolist())\n            feats.extend(vals.std(axis=0).astype(float).tolist())\n            feats.extend(np.percentile(vals, 85, axis=0).astype(float).tolist())\n    weighted = pattern * valid[:, :, None].astype(np.float32)\n    denom_y = np.maximum(1.0, valid.sum(axis=0).astype(np.float32))\n    denom_x = np.maximum(1.0, valid.sum(axis=1).astype(np.float32))\n    proj_x = weighted[:, :, 0].sum(axis=0) / denom_y\n    proj_y = weighted[:, :, 0].sum(axis=1) / denom_x\n    proj_x = cv2.resize(proj_x.reshape(1, -1), (48, 1), interpolation=cv2.INTER_AREA).reshape(-1)\n    proj_y = cv2.resize(proj_y.reshape(-1, 1), (1, 32), interpolation=cv2.INTER_AREA).reshape(-1)\n    feats.extend(proj_x.astype(float).tolist())\n    feats.extend(proj_y.astype(float).tolist())\n    return normalize(np.asarray(feats, dtype=np.float32))\n\n\ndef detect_pattern_points(pattern: np.ndarray, mask: np.ndarray, species: str) -> np.ndarray:\n    heat = pattern[:, :, 0].copy()\n    valid = mask > 0\n    if valid.mean() < 0.025:\n        return np.zeros((0, 4), dtype=np.float32)\n    vals = heat[valid]\n    if vals.size < 32:\n        return np.zeros((0, 4), dtype=np.float32)\n    pct = 82.0 if species in {\"LynxID2025\", \"TexasHornedLizards\"} else 88.0\n    thr = max(float(np.percentile(vals, pct)), float(vals.mean() + 0.55 * vals.std()))\n    binary = np.zeros(heat.shape, dtype=np.uint8)\n    binary[(heat >= thr) & valid] = 255\n    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n    n, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, 8)\n    h, w = heat.shape[:2]\n    total = float(h * w)\n    pts: list[list[float]] = []\n    min_area = max(2.0, total * (0.00006 if species != \"SeaTurtleID2022\" else 0.00012))\n    max_area = total * (0.020 if species != \"SeaTurtleID2022\" else 0.045)\n    for idx in range(1, n):\n        area = float(stats[idx, cv2.CC_STAT_AREA])\n        if area < min_area or area > max_area:\n            continue\n        bw = max(1.0, float(stats[idx, cv2.CC_STAT_WIDTH]))\n        bh = max(1.0, float(stats[idx, cv2.CC_STAT_HEIGHT]))\n        if max(bw / bh, bh / bw) > 7.5:\n            continue\n        cx, cy = centroids[idx]\n        strength = float(heat[labels == idx].mean())\n        pts.append([float(cx) / max(1, w - 1), float(cy) / max(1, h - 1), area / total, strength])\n    if not pts:\n        return np.zeros((0, 4), dtype=np.float32)\n    pts_arr = np.asarray(pts, dtype=np.float32)\n    order = np.argsort(-pts_arr[:, 3])\n    return pts_arr[order[:220]]\n\n\ndef build_template(row: dict, cfg: AnatomyConfig, max_side: int) -> AnatomyItem:\n    rgb, mask, quality = read_rgb_mask(row, max_side)\n    debug: dict = {}\n    if cfg.species == \"SalamanderID2025\":\n        template_rgb, template_mask, debug = unwrap_centerline_strip(rgb, mask, cfg.target_size)\n    elif cfg.species == \"TexasHornedLizards\":\n        template_rgb, template_mask, debug = pca_align_template(rgb, mask, cfg.target_size, \"vertical\", 0.08)\n    elif cfg.species == \"LynxID2025\":\n        template_rgb, template_mask, debug = pca_align_template(rgb, mask, cfg.target_size, \"horizontal\", 0.08)\n    else:\n        template_rgb, template_mask, debug = pca_align_template(rgb, mask, cfg.target_size, \"horizontal\", 0.04)\n    template_mask = apply_species_template_mask(template_mask, cfg.species)\n    if float(template_mask.mean() / 255.0) < 0.025:\n        template_mask = np.where(cv2.cvtColor(template_rgb, cv2.COLOR_RGB2GRAY) >= 0, 255, 0).astype(np.uint8)\n        quality *= 0.70\n    pattern = pattern_channels(template_rgb, template_mask, cfg.species)\n    vector = grid_descriptor(pattern, template_mask, cfg.grid)\n    points = detect_pattern_points(pattern, template_mask, cfg.species)\n    quality *= min(1.0, 0.65 + 0.35 * float(template_mask.mean() / 255.0) / max(0.08, cfg.min_overlap))\n    return AnatomyItem(\n        image_id=int(row[\"image_id\"]),\n        row_idx=int(row.get(\"row_idx\", row[\"image_id\"])),\n        identity=str(row.get(\"identity\", \"\") or \"\"),\n        split=str(row.get(\"split\", \"\")),\n        species=cfg.species,\n        orientation=str(row.get(\"orientation\", \"unknown\") or \"unknown\").lower(),\n        source_path=str(row.get(\"source_path\", \"\")),\n        sam_view_path=str(row.get(\"sam_view_path\", \"\") or \"\"),\n        view_path=str(row.get(\"view_path\", row.get(\"source_path\", \"\"))),\n        mask_path=str(row.get(\"mask_path\", \"\") or \"\"),\n        view_source=str(row.get(\"view_source\", \"\")),\n        quality=float(quality),\n        template_rgb=template_rgb,\n        template_mask=template_mask,\n        pattern=pattern,\n        vector=vector,\n        points=points,\n        debug=debug,\n    )\n\n\ndef transform_item_arrays(\n    pattern: np.ndarray,\n    mask: np.ndarray,\n    points: np.ndarray,\n    transform: str,\n) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    pat = pattern\n    m = mask\n    pts = points.copy()\n    if transform in {\"rev_x\", \"rev_xy\"}:\n        pat = pat[:, ::-1, :]\n        m = m[:, ::-1]\n        if len(pts):\n            pts[:, 0] = 1.0 - pts[:, 0]\n    if transform in {\"rev_y\", \"rev_xy\"}:\n        pat = pat[::-1, :, :]\n        m = m[::-1, :]\n        if len(pts):\n            pts[:, 1] = 1.0 - pts[:, 1]\n    return pat, m, pts\n\n\ndef shifted_slices(shape: tuple[int, int], dx: int, dy: int):\n    h, w = shape\n    xa1 = max(0, dx)\n    xb1 = max(0, -dx)\n    ya1 = max(0, dy)\n    yb1 = max(0, -dy)\n    ww = w - abs(dx)\n    hh = h - abs(dy)\n    if ww <= 4 or hh <= 4:\n        return None\n    return (slice(ya1, ya1 + hh), slice(xa1, xa1 + ww)), (slice(yb1, yb1 + hh), slice(xb1, xb1 + ww))\n\n\ndef masked_template_similarity(\n    pat_a: np.ndarray,\n    mask_a: np.ndarray,\n    pat_b: np.ndarray,\n    mask_b: np.ndarray,\n    dx: int,\n    dy: int,\n) -> tuple[float, float]:\n    slices = shifted_slices(mask_a.shape[:2], dx, dy)\n    if slices is None:\n        return 0.0, 0.0\n    sa, sb = slices\n    ma = mask_a[sa] > 0\n    mb = mask_b[sb] > 0\n    overlap_mask = ma & mb\n    overlap = float(overlap_mask.mean()) if overlap_mask.size else 0.0\n    if overlap < 0.025 or int(overlap_mask.sum()) < 30:\n        return 0.0, overlap\n    sub_a = pat_a[sa]\n    sub_b = pat_b[sb]\n    comp_a = 0.66 * sub_a[:, :, 0] + 0.24 * sub_a[:, :, 1] + 0.10 * sub_a[:, :, 2]\n    comp_b = 0.66 * sub_b[:, :, 0] + 0.24 * sub_b[:, :, 1] + 0.10 * sub_b[:, :, 2]\n    vals_a = comp_a[overlap_mask].astype(np.float32)\n    vals_b = comp_b[overlap_mask].astype(np.float32)\n    if vals_a.size == 0:\n        return 0.0, overlap\n    am = vals_a - float(vals_a.mean())\n    bm = vals_b - float(vals_b.mean())\n    denom = float(np.linalg.norm(am) * np.linalg.norm(bm))\n    corr = float(np.dot(am, bm) / denom) if denom > 1e-6 else 0.0\n    l1 = float(1.0 - np.mean(np.abs(vals_a - vals_b)))\n    score = 0.62 * ((corr + 1.0) * 0.5) + 0.38 * np.clip(l1, 0.0, 1.0)\n    return float(np.clip(score, 0.0, 1.0)), overlap\n\n\ndef point_chamfer_score(points_a: np.ndarray, points_b: np.ndarray, dx_norm: float, dy_norm: float) -> float:\n    if len(points_a) < 3 or len(points_b) < 3:\n        return 0.0\n    a = points_a[:, :2].astype(np.float32)\n    b = points_b[:, :2].astype(np.float32).copy()\n    b[:, 0] += dx_norm\n    b[:, 1] += dy_norm\n    diff = a[:, None, :] - b[None, :, :]\n    dist = np.sqrt(np.maximum(0.0, (diff * diff).sum(axis=2)))\n    da = dist.min(axis=1)\n    db = dist.min(axis=0)\n    keep_a = np.argsort(da)[: min(len(da), 80)]\n    keep_b = np.argsort(db)[: min(len(db), 80)]\n    mean_d = 0.5 * (float(da[keep_a].mean()) + float(db[keep_b].mean()))\n    return float(np.exp(-mean_d / 0.055))\n\n\ndef allowed_transforms(a: AnatomyItem, b: AnatomyItem) -> list[str]:\n    if a.species == \"SalamanderID2025\":\n        return [\"identity\", \"rev_x\"]\n    if a.species == \"TexasHornedLizards\":\n        return [\"identity\", \"rev_y\", \"rev_xy\"]\n    if a.species == \"SeaTurtleID2022\":\n        return [\"identity\", \"rev_x\", \"rev_y\"]\n    if a.species == \"LynxID2025\":\n        sides = {\"left\", \"right\"}\n        if a.orientation in sides and b.orientation in sides and a.orientation != b.orientation:\n            return [\"identity\"]\n        if a.orientation == \"unknown\" or b.orientation == \"unknown\":\n            return [\"identity\", \"rev_x\"]\n    return [\"identity\"]\n\n\ndef orientation_penalty(a: AnatomyItem, b: AnatomyItem) -> tuple[float, str]:\n    if a.species != \"LynxID2025\":\n        return 1.0, \"not_applicable\"\n    sides = {\"left\", \"right\"}\n    if a.orientation in sides and b.orientation in sides and a.orientation != b.orientation:\n        return 0.55, \"lynx_opposite_flank\"\n    if a.orientation in {\"front\", \"back\"} or b.orientation in {\"front\", \"back\"}:\n        return 0.84, \"lynx_weak_pose\"\n    return 1.0, \"lynx_same_or_unknown\"\n\n\ndef template_match_score(a: AnatomyItem, b: AnatomyItem, cfg: AnatomyConfig) -> dict:\n    penalty, orient_rule = orientation_penalty(a, b)\n    desc_cos = float(np.dot(a.vector, b.vector))\n    best = {\n        \"map_score\": 0.0,\n        \"overlap\": 0.0,\n        \"point_score\": 0.0,\n        \"transform\": \"none\",\n        \"dx\": 0,\n        \"dy\": 0,\n    }\n    h, w = a.template_mask.shape[:2]\n    shifts = cfg.shift_px\n    if cfg.species == \"SalamanderID2025\":\n        shift_pairs = [(dx, dy) for dx in shifts for dy in (-4, 0, 4)]\n    else:\n        shift_pairs = [(dx, dy) for dx in shifts for dy in shifts]\n    for transform in allowed_transforms(a, b):\n        pat_b, mask_b, pts_b = transform_item_arrays(b.pattern, b.template_mask, b.points, transform)\n        for dx, dy in shift_pairs:\n            map_score, overlap = masked_template_similarity(a.pattern, a.template_mask, pat_b, mask_b, dx, dy)\n            if overlap < 0.025:\n                continue\n            point_score = point_chamfer_score(a.points, pts_b, dx / max(1, w - 1), dy / max(1, h - 1))\n            if cfg.species == \"SalamanderID2025\":\n                fused = 0.86 * map_score + 0.14 * max(0.0, desc_cos)\n            elif cfg.species in {\"LynxID2025\", \"TexasHornedLizards\"}:\n                fused = 0.58 * map_score + 0.30 * point_score + 0.12 * max(0.0, desc_cos)\n            else:\n                fused = 0.72 * map_score + 0.10 * point_score + 0.18 * max(0.0, desc_cos)\n            # Reward meaningful overlap, but still allow corrupted/missing body parts\n            # when the visible fingerprint region matches strongly.\n            overlap_factor = min(1.0, 0.70 + 0.30 * overlap / max(0.08, cfg.min_overlap))\n            fused = float(fused * overlap_factor * penalty * min(a.quality, b.quality))\n            if fused > best.get(\"score\", -1.0):\n                best = {\n                    \"score\": fused,\n                    \"map_score\": float(map_score),\n                    \"overlap\": float(overlap),\n                    \"point_score\": float(point_score),\n                    \"transform\": transform,\n                    \"dx\": int(dx),\n                    \"dy\": int(dy),\n                }\n    if \"score\" not in best:\n        best[\"score\"] = 0.0\n    best[\"descriptor_cosine\"] = desc_cos\n    best[\"orientation_rule\"] = orient_rule\n    best[\"quality_a\"] = float(a.quality)\n    best[\"quality_b\"] = float(b.quality)\n    best[\"points_a\"] = int(len(a.points))\n    best[\"points_b\"] = int(len(b.points))\n    return best\n\n\ndef shortlist_pairs(items: list[AnatomyItem], cfg: AnatomyConfig, top_k_scale: float, pair_budget: int) -> list[tuple[int, int]]:\n    if len(items) < 2:\n        return []\n    vectors = np.stack([it.vector for it in items]).astype(np.float32)\n    sim = vectors @ vectors.T\n    np.fill_diagonal(sim, -np.inf)\n    ids = [it.image_id for it in items]\n    item_by_id = {it.image_id: it for it in items}\n    top_k = max(1, int(round(cfg.top_k * top_k_scale)))\n    pairs: dict[tuple[int, int], float] = {}\n    for i, image_id in enumerate(ids):\n        k = min(top_k, len(ids) - 1)\n        if k <= 0:\n            continue\n        idx = np.argpartition(-sim[i], kth=k - 1)[:k]\n        for j in idx:\n            other_id = ids[int(j)]\n            if other_id == image_id:\n                continue\n            a, b = (image_id, other_id) if image_id < other_id else (other_id, image_id)\n            ita = item_by_id[a]\n            itb = item_by_id[b]\n            if cfg.species == \"LynxID2025\":\n                sides = {\"left\", \"right\"}\n                if ita.orientation in sides and itb.orientation in sides and ita.orientation != itb.orientation:\n                    if float(sim[i, int(j)]) < 0.78:\n                        continue\n            pairs[(a, b)] = max(pairs.get((a, b), -1.0), float(sim[i, int(j)]))\n    ranked = sorted(pairs.items(), key=lambda kv: -kv[1])\n    if pair_budget and len(ranked) > pair_budget:\n        ranked = ranked[:pair_budget]\n    return [p for p, _ in ranked]\n\n\ndef extract_items(rows: pd.DataFrame, cfg: AnatomyConfig, args: argparse.Namespace, split_name: str) -> tuple[list[AnatomyItem], dict]:\n    records = rows.sort_values(\"image_id\").to_dict(\"records\")\n    limit_attr = \"max_test_images_per_species\" if split_name == \"test\" else \"max_train_images_per_species\"\n    limit = int(getattr(args, limit_attr, 0) or 0)\n    if limit > 0 and len(records) > limit:\n        records = records[:limit]\n    if args.smoke:\n        records = records[: min(len(records), 20 if split_name == \"test\" else 28)]\n    items: list[AnatomyItem] = []\n    failures = 0\n    for idx, row in enumerate(records, start=1):\n        try:\n            items.append(build_template(row, cfg, args.max_side))\n        except Exception as exc:\n            failures += 1\n            w, h = cfg.target_size\n            blank_rgb = np.full((h, w, 3), BACKGROUND, dtype=np.uint8)\n            blank_mask = np.zeros((h, w), dtype=np.uint8)\n            blank_pattern = np.zeros((h, w, 3), dtype=np.float32)\n            items.append(\n                AnatomyItem(\n                    image_id=int(row[\"image_id\"]),\n                    row_idx=int(row.get(\"row_idx\", row[\"image_id\"])),\n                    identity=str(row.get(\"identity\", \"\") or \"\"),\n                    split=str(row.get(\"split\", split_name)),\n                    species=cfg.species,\n                    orientation=str(row.get(\"orientation\", \"unknown\") or \"unknown\").lower(),\n                    source_path=str(row.get(\"source_path\", \"\")),\n                    sam_view_path=str(row.get(\"sam_view_path\", \"\") or \"\"),\n                    view_path=str(row.get(\"view_path\", row.get(\"source_path\", \"\"))),\n                    mask_path=str(row.get(\"mask_path\", \"\") or \"\"),\n                    view_source=\"failed\",\n                    quality=0.0,\n                    template_rgb=blank_rgb,\n                    template_mask=blank_mask,\n                    pattern=blank_pattern,\n                    vector=np.zeros_like(grid_descriptor(blank_pattern, blank_mask, cfg.grid)),\n                    points=np.zeros((0, 4), dtype=np.float32),\n                    debug={\"error\": str(exc)},\n                )\n            )\n            print(f\"[warn] {cfg.species} {split_name}: failed image_id={row.get('image_id')}: {exc}\")\n        if idx % 100 == 0:\n            print(f\"[{cfg.species} {split_name}] templates {idx}/{len(records)}\")\n    info = {\n        \"species\": cfg.species,\n        \"split\": split_name,\n        \"n_items\": int(len(items)),\n        \"failures\": int(failures),\n        \"sam_clean_primary\": int(sum(it.view_source == \"sam_clean_primary\" for it in items)),\n        \"mean_quality\": float(np.mean([it.quality for it in items])) if items else 0.0,\n        \"median_points\": float(np.median([len(it.points) for it in items])) if items else 0.0,\n        \"mean_template_coverage\": float(np.mean([it.template_mask.mean() / 255.0 for it in items])) if items else 0.0,\n    }\n    return items, info\n\n\ndef score_item_pairs(\n    species: str,\n    items: list[AnatomyItem],\n    pairs: list[tuple[int, int]],\n    cfg: AnatomyConfig,\n) -> pd.DataFrame:\n    by_id = {it.image_id: it for it in items}\n    rows = []\n    for idx, (a_id, b_id) in enumerate(pairs, start=1):\n        a = by_id.get(int(a_id))\n        b = by_id.get(int(b_id))\n        if a is None or b is None:\n            continue\n        score = template_match_score(a, b, cfg)\n        same_identity = bool(a.identity and b.identity and a.identity == b.identity)\n        rows.append(\n            {\n                \"species\": species,\n                \"image_id_a\": int(a_id),\n                \"image_id_b\": int(b_id),\n                \"identity_a\": a.identity,\n                \"identity_b\": b.identity,\n                \"same_identity\": same_identity,\n                \"orientation_a\": a.orientation,\n                \"orientation_b\": b.orientation,\n                **score,\n            }\n        )\n        if idx % 5000 == 0:\n            print(f\"[{species}] scored {idx}/{len(pairs)} template pairs\")\n    return pd.DataFrame(rows)\n\n\ndef sample_train_rows(rows: pd.DataFrame, max_images: int, max_per_identity: int) -> pd.DataFrame:\n    labeled = rows[rows[\"identity\"].astype(str).str.len().gt(0)].copy()\n    if labeled.empty:\n        return labeled\n    rng = random.Random(SEED)\n    chosen = []\n    for _, group in labeled.groupby(\"identity\"):\n        recs = group.sort_values(\"image_id\").to_dict(\"records\")\n        if len(recs) > max_per_identity:\n            recs = rng.sample(recs, max_per_identity)\n        chosen.extend(recs)\n    chosen.sort(key=lambda r: int(r[\"image_id\"]))\n    if max_images > 0 and len(chosen) > max_images:\n        chosen = rng.sample(chosen, max_images)\n        chosen.sort(key=lambda r: int(r[\"image_id\"]))\n    return pd.DataFrame(chosen)\n\n\ndef sample_validation_pairs(items: list[AnatomyItem], pair_budget: int) -> list[tuple[int, int]]:\n    rng = random.Random(SEED)\n    by_identity: dict[str, list[int]] = {}\n    ids = []\n    identities = {}\n    for it in items:\n        if not it.identity:\n            continue\n        by_identity.setdefault(it.identity, []).append(it.image_id)\n        ids.append(it.image_id)\n        identities[it.image_id] = it.identity\n    positives = []\n    for members in by_identity.values():\n        if len(members) < 2:\n            continue\n        combos = list(itertools.combinations(sorted(members), 2))\n        if len(combos) > 10:\n            combos = rng.sample(combos, 10)\n        positives.extend(combos)\n    if len(positives) > pair_budget // 2:\n        positives = rng.sample(positives, max(1, pair_budget // 2))\n    negatives = set()\n    target_neg = min(pair_budget - len(positives), max(len(positives) * 2, 500 if positives else pair_budget))\n    attempts = 0\n    while len(negatives) < target_neg and attempts < target_neg * 25 and len(ids) >= 2:\n        a, b = rng.sample(ids, 2)\n        attempts += 1\n        if identities.get(a) == identities.get(b):\n            continue\n        key = (a, b) if a < b else (b, a)\n        negatives.add(key)\n    pairs = list(positives) + sorted(negatives)\n    if len(pairs) > pair_budget:\n        pairs = rng.sample(pairs, pair_budget)\n    return [(int(a), int(b)) for a, b in pairs]\n\n\ndef auc_rank(y_true: np.ndarray, scores: np.ndarray) -> float:\n    y = y_true.astype(bool)\n    n_pos = int(y.sum())\n    n_neg = int((~y).sum())\n    if n_pos == 0 or n_neg == 0:\n        return float(\"nan\")\n    order = np.argsort(scores)\n    ranks = np.empty_like(order, dtype=np.float64)\n    ranks[order] = np.arange(1, len(scores) + 1, dtype=np.float64)\n    pos_rank_sum = float(ranks[y].sum())\n    return float((pos_rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))\n\n\ndef validation_summary(pair_scores: pd.DataFrame, cfg: AnatomyConfig) -> dict:\n    if pair_scores.empty or \"same_identity\" not in pair_scores.columns:\n        return {\"species\": cfg.species, \"auc\": float(\"nan\"), \"n_pairs\": int(len(pair_scores))}\n    y = pair_scores[\"same_identity\"].astype(bool).to_numpy()\n    scores = pair_scores[\"score\"].astype(float).to_numpy()\n    summary = {\n        \"species\": cfg.species,\n        \"n_pairs\": int(len(pair_scores)),\n        \"n_positive\": int(y.sum()),\n        \"n_negative\": int((~y).sum()),\n        \"auc\": auc_rank(y, scores),\n    }\n    for profile in [\"strict\", \"balanced\", \"aggressive\", \"force\"]:\n        thr = getattr(cfg, f\"{profile}_thr\")\n        pred = (scores >= thr) & (pair_scores[\"overlap\"].astype(float).to_numpy() >= cfg.min_overlap)\n        tp = int((pred & y).sum())\n        fp = int((pred & (~y)).sum())\n        fn = int(((~pred) & y).sum())\n        precision = tp / max(1, tp + fp)\n        recall = tp / max(1, tp + fn)\n        summary[f\"{profile}_thr\"] = float(thr)\n        summary[f\"{profile}_accepted\"] = int(pred.sum())\n        summary[f\"{profile}_precision\"] = float(precision)\n        summary[f\"{profile}_recall\"] = float(recall)\n    return summary\n\n\ndef threshold_for_profile(cfg: AnatomyConfig, profile: str) -> float:\n    if profile not in {\"strict\", \"balanced\", \"aggressive\", \"force\"}:\n        raise ValueError(f\"Unknown profile: {profile}\")\n    return float(getattr(cfg, f\"{profile}_thr\"))\n\n\ndef accepted_pair_rows(pair_scores: pd.DataFrame, cfg: AnatomyConfig, profile: str) -> pd.DataFrame:\n    if pair_scores.empty:\n        return pair_scores.copy()\n    thr = threshold_for_profile(cfg, profile)\n    g = pair_scores.copy()\n    ok = (g[\"score\"].astype(float) >= thr) & (g[\"overlap\"].astype(float) >= cfg.min_overlap)\n    if cfg.species == \"LynxID2025\" and profile != \"force\":\n        ok &= ~g[\"orientation_rule\"].astype(str).eq(\"lynx_opposite_flank\")\n    g = g[ok].sort_values(\"score\", ascending=False).copy()\n    return g\n\n\ndef cluster_from_pairs(items: list[AnatomyItem], pair_scores: pd.DataFrame, cfg: AnatomyConfig, profile: str) -> dict[int, str]:\n    ids = [it.image_id for it in items]\n    uf = UnionFind(ids)\n    accepted = accepted_pair_rows(pair_scores, cfg, profile)\n    for row in accepted.itertuples(index=False):\n        uf.union(int(row.image_id_a), int(row.image_id_b))\n    comp_order: dict[int, int] = {}\n    labels: dict[int, str] = {}\n    for image_id in sorted(ids):\n        comp = uf.find(image_id)\n        if comp not in comp_order:\n            comp_order[comp] = len(comp_order)\n        labels[image_id] = f\"cluster_{cfg.species}_anatomy_{profile}_{comp_order[comp]}\"\n    return labels\n\n\ndef summarize_labels(labels: dict[int, str], cfg: AnatomyConfig, profile: str, pair_scores: pd.DataFrame) -> dict:\n    counts = pd.Series(list(labels.values())).value_counts()\n    accepted = accepted_pair_rows(pair_scores, cfg, profile)\n    return {\n        \"variant\": profile,\n        \"species\": cfg.species,\n        \"n_images\": int(len(labels)),\n        \"n_clusters\": int(counts.shape[0]) if not counts.empty else 0,\n        \"singletons\": int((counts == 1).sum()) if not counts.empty else 0,\n        \"max_cluster_size\": int(counts.max()) if not counts.empty else 0,\n        \"accepted_edges\": int(len(accepted)),\n        \"mean_accepted_score\": float(accepted[\"score\"].mean()) if not accepted.empty else 0.0,\n        \"max_accepted_score\": float(accepted[\"score\"].max()) if not accepted.empty else 0.0,\n    }\n\n\ndef save_submission(\n    sample_submission: pd.DataFrame,\n    labels_by_species: dict[str, dict[int, str]],\n    profile: str,\n    output_path: Path,\n) -> pd.DataFrame:\n    label_map: dict[int, str] = {}\n    for labels in labels_by_species.values():\n        label_map.update(labels)\n    sub = sample_submission.copy()\n    sub[\"image_id\"] = sub[\"image_id\"].astype(int)\n    sub[\"cluster\"] = sub[\"image_id\"].map(lambda i: label_map.get(int(i), f\"cluster_missing_{int(i)}\"))\n    sub.to_csv(output_path, index=False)\n    return sub\n\n\ndef thumb(path: str, size: tuple[int, int]) -> Image.Image:\n    try:\n        img = Image.open(path).convert(\"RGB\")\n    except Exception:\n        img = Image.new(\"RGB\", size, (25, 25, 25))\n    img.thumbnail(size, Image.Resampling.BILINEAR)\n    canvas = Image.new(\"RGB\", size, (20, 20, 20))\n    canvas.paste(img, ((size[0] - img.width) // 2, (size[1] - img.height) // 2))\n    return canvas\n\n\ndef pattern_to_image(pattern: np.ndarray, mask: np.ndarray) -> Image.Image:\n    pat = np.clip(pattern.copy(), 0, 1)\n    if pat.shape[2] >= 3:\n        arr = (pat[:, :, :3] * 255).astype(np.uint8)\n    else:\n        arr = np.repeat((pat[:, :, :1] * 255).astype(np.uint8), 3, axis=2)\n    arr[mask == 0] = BACKGROUND\n    return Image.fromarray(arr)\n\n\ndef mask_overlay(rgb: np.ndarray, mask: np.ndarray, points: np.ndarray) -> Image.Image:\n    canvas = rgb.copy()\n    dim = (canvas.astype(np.float32) * 0.45 + BACKGROUND.astype(np.float32) * 0.55).astype(np.uint8)\n    canvas[mask == 0] = dim[mask == 0]\n    contours, _ = cv2.findContours(np.where(mask > 0, 255, 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    cv2.drawContours(canvas, contours, -1, (40, 255, 80), 2)\n    h, w = mask.shape[:2]\n    for p in points[:140]:\n        x = int(round(float(p[0]) * max(1, w - 1)))\n        y = int(round(float(p[1]) * max(1, h - 1)))\n        cv2.circle(canvas, (x, y), 2, (255, 30, 30), 1, cv2.LINE_AA)\n    return Image.fromarray(canvas)\n\n\ndef save_template_preview(items: list[AnatomyItem], out_path: Path, limit: int) -> None:\n    chosen = items[:limit]\n    if not chosen:\n        return\n    tile_w, tile_h = 210, 136\n    label_h = 22\n    cols = 5\n    canvas = Image.new(\"RGB\", (cols * tile_w, len(chosen) * (tile_h + label_h)), (18, 18, 18))\n    draw = ImageDraw.Draw(canvas)\n    for r, item in enumerate(chosen):\n        y = r * (tile_h + label_h)\n        sam_path = item.sam_view_path or item.view_path\n        panels = [\n            thumb(item.source_path, (tile_w, tile_h)),\n            thumb(sam_path, (tile_w, tile_h)),\n            mask_overlay(item.template_rgb, item.template_mask, item.points),\n            Image.fromarray(item.template_rgb),\n            pattern_to_image(item.pattern, item.template_mask),\n        ]\n        labels = [\n            f\"orig {item.image_id}\",\n            item.view_source[:22],\n            \"template mask + points\",\n            \"canonical anatomy\",\n            \"pattern map\",\n        ]\n        for c, panel in enumerate(panels):\n            panel = panel.copy()\n            panel.thumbnail((tile_w, tile_h), Image.Resampling.BILINEAR)\n            x = c * tile_w\n            draw.text((x + 5, y + 4), labels[c], fill=(245, 240, 145))\n            canvas.paste(panel, (x + (tile_w - panel.width) // 2, y + label_h + (tile_h - panel.height) // 2))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(out_path, quality=92)\n\n\ndef save_pair_preview(\n    items: list[AnatomyItem],\n    pair_scores: pd.DataFrame,\n    cfg: AnatomyConfig,\n    profile: str,\n    out_path: Path,\n    limit: int,\n) -> None:\n    accepted = accepted_pair_rows(pair_scores, cfg, profile).head(limit)\n    if accepted.empty:\n        return\n    by_id = {it.image_id: it for it in items}\n    panel_w, panel_h = 920, 260\n    rows = []\n    for edge in accepted.to_dict(\"records\"):\n        a = by_id.get(int(edge[\"image_id_a\"]))\n        b = by_id.get(int(edge[\"image_id_b\"]))\n        if a is None or b is None:\n            continue\n        pa = pattern_to_image(a.pattern, a.template_mask)\n        pb = pattern_to_image(b.pattern, b.template_mask)\n        pa.thumbnail((440, 205), Image.Resampling.BILINEAR)\n        pb.thumbnail((440, 205), Image.Resampling.BILINEAR)\n        row_img = Image.new(\"RGB\", (panel_w, panel_h), (18, 18, 18))\n        draw = ImageDraw.Draw(row_img)\n        text = (\n            f\"{cfg.species} {profile}: {a.image_id} vs {b.image_id} \"\n            f\"score={float(edge['score']):.3f} map={float(edge['map_score']):.3f} \"\n            f\"pts={float(edge['point_score']):.3f} overlap={float(edge['overlap']):.3f} \"\n            f\"{edge['transform']} dx={edge['dx']} dy={edge['dy']}\"\n        )\n        draw.text((6, 5), text, fill=(255, 240, 120))\n        row_img.paste(pa, (8, 42))\n        row_img.paste(pb, (470, 42))\n        rows.append(row_img)\n    if not rows:\n        return\n    canvas = Image.new(\"RGB\", (panel_w, panel_h * len(rows)), (18, 18, 18))\n    for idx, row_img in enumerate(rows):\n        canvas.paste(row_img, (0, idx * panel_h))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(out_path, quality=92)\n\n\ndef update_experiment_log(output_root: Path, summary: dict) -> None:\n    log_path = Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\current_wildfusion_graph_v20260423\\experiment_log.json\")\n    if not log_path.exists():\n        return\n    try:\n        log = json.loads(log_path.read_text(encoding=\"utf-8\"))\n    except Exception:\n        return\n    entry = {\n        \"run_id\": VERSION,\n        \"date\": \"2026-04-26\",\n        \"status\": \"notebook_built_ready_for_kaggle\",\n        \"goal\": \"Independent species-specific anatomy-template registration: SAM-clean foreground, canonical body maps, occlusion-aware fingerprint overlap, and template-space clustering.\",\n        \"notebook\": \"notebooks/AnimalCLEF2026_Anatomy_Template_Registration_v20260426.ipynb\",\n        \"output_root\": str(output_root),\n        \"summary\": summary,\n    }\n    if isinstance(log, dict):\n        runs = log.setdefault(\"runs\", [])\n        log[\"runs\"] = [r for r in runs if not (isinstance(r, dict) and r.get(\"run_id\") == VERSION)]\n        log[\"runs\"].append(entry)\n        log_path.write_text(json.dumps(log, indent=2), encoding=\"utf-8\")\n\n\ndef main() -> None:\n    random.seed(SEED)\n    np.random.seed(SEED)\n    args = parse_args()\n    if args.smoke:\n        args.max_train_images_per_species = min(args.max_train_images_per_species, 40)\n        args.max_test_images_per_species = args.max_test_images_per_species or 18\n        args.train_pair_budget_per_species = min(args.train_pair_budget_per_species, 400)\n        args.test_pair_budget_per_species = min(args.test_pair_budget_per_species, 650)\n        args.top_k_scale = min(args.top_k_scale, 0.35)\n        args.save_visualizations = True\n\n    data_root = find_data_root(args.data_root)\n    sam_manifest = find_sam_manifest(args.sam_manifest)\n    metadata, manifest_info = prepare_metadata(data_root, sam_manifest)\n    metadata = metadata[metadata[\"species_id\"].isin(core.SPECIES_ORDER)].copy()\n    output_root = Path(args.output_root) if args.output_root else Path.cwd() / f\"animalclef_{VERSION}\"\n    reports_dir = output_root / \"reports\"\n    submissions_dir = output_root / \"submissions\"\n    visual_dir = output_root / \"visualizations\"\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    submissions_dir.mkdir(parents=True, exist_ok=True)\n    if args.save_visualizations:\n        visual_dir.mkdir(parents=True, exist_ok=True)\n\n    selected_species = [s for s in args.species if s in CONFIGS]\n    sample_submission = pd.read_csv(data_root / \"sample_submission.csv\")\n    sample_submission[\"image_id\"] = sample_submission[\"image_id\"].astype(int)\n    test_rows_all = metadata[metadata[\"split\"].eq(\"test\")].copy()\n    train_rows_all = metadata[metadata[\"split\"].eq(\"train\")].copy()\n\n    print(f\"VERSION={VERSION}\")\n    print(f\"data_root={data_root}\")\n    print(f\"sam_manifest={sam_manifest}\")\n    print(f\"output_root={output_root}\")\n    print(json.dumps(manifest_info, indent=2))\n\n    test_items_by_species: dict[str, list[AnatomyItem]] = {}\n    test_pairs_by_species: dict[str, pd.DataFrame] = {}\n    extraction_infos: list[dict] = []\n    validation_rows: list[dict] = []\n\n    for species in selected_species:\n        cfg = CONFIGS[species]\n        print(f\"\\n=== {species}: test anatomy templates ===\")\n        test_rows = test_rows_all[test_rows_all[\"species_id\"].eq(species)].copy()\n        test_items, info = extract_items(test_rows, cfg, args, \"test\")\n        extraction_infos.append(info)\n        test_items_by_species[species] = test_items\n        pairs = shortlist_pairs(test_items, cfg, args.top_k_scale, args.test_pair_budget_per_species)\n        print(f\"[{species}] shortlisted {len(pairs)} test pairs\")\n        pair_scores = score_item_pairs(species, test_items, pairs, cfg)\n        test_pairs_by_species[species] = pair_scores\n        pair_scores.to_csv(reports_dir / f\"{VERSION}_{species}_test_pair_scores.csv\", index=False)\n        if args.save_visualizations:\n            save_template_preview(test_items, visual_dir / f\"{VERSION}_{species}_template_preview.jpg\", args.visual_limit)\n            for profile in args.profiles:\n                save_pair_preview(\n                    test_items,\n                    pair_scores,\n                    cfg,\n                    profile,\n                    visual_dir / f\"{VERSION}_{species}_{profile}_top_matches.jpg\",\n                    max(3, args.visual_limit // 2),\n                )\n\n        if args.skip_train_validation or species == \"TexasHornedLizards\":\n            continue\n        print(f\"\\n=== {species}: train-label validation ===\")\n        train_rows = train_rows_all[train_rows_all[\"species_id\"].eq(species)].copy()\n        train_rows = sample_train_rows(train_rows, args.max_train_images_per_species, args.max_train_per_identity)\n        if train_rows.empty:\n            continue\n        train_items, train_info = extract_items(train_rows, cfg, args, \"train\")\n        extraction_infos.append(train_info)\n        val_pairs = sample_validation_pairs(train_items, args.train_pair_budget_per_species)\n        print(f\"[{species}] validation pairs {len(val_pairs)}\")\n        val_scores = score_item_pairs(species, train_items, val_pairs, cfg)\n        val_scores.to_csv(reports_dir / f\"{VERSION}_{species}_train_validation_pairs.csv\", index=False)\n        validation_rows.append(validation_summary(val_scores, cfg))\n\n    all_test_pairs = pd.concat([df for df in test_pairs_by_species.values() if not df.empty], ignore_index=True) if test_pairs_by_species else pd.DataFrame()\n    if not all_test_pairs.empty:\n        all_test_pairs.to_csv(reports_dir / f\"{VERSION}_all_test_pair_scores.csv\", index=False)\n\n    labels_by_profile: dict[str, dict[str, dict[int, str]]] = {profile: {} for profile in args.profiles}\n    candidate_rows: list[dict] = []\n    for profile in args.profiles:\n        for species in selected_species:\n            cfg = CONFIGS[species]\n            items = test_items_by_species.get(species, [])\n            pair_scores = test_pairs_by_species.get(species, pd.DataFrame())\n            labels = cluster_from_pairs(items, pair_scores, cfg, profile)\n            labels_by_profile[profile][species] = labels\n            candidate_rows.append(summarize_labels(labels, cfg, profile, pair_scores))\n        sub_path = submissions_dir / f\"submission_{VERSION}_{profile}.csv\"\n        save_submission(sample_submission, labels_by_profile[profile], profile, sub_path)\n        print(f\"wrote {sub_path}\")\n\n    extraction_report = pd.DataFrame(extraction_infos)\n    validation_report = pd.DataFrame(validation_rows)\n    candidate_report = pd.DataFrame(candidate_rows)\n    extraction_report.to_csv(reports_dir / f\"{VERSION}_extraction_report.csv\", index=False)\n    validation_report.to_csv(reports_dir / f\"{VERSION}_train_validation_report.csv\", index=False)\n    candidate_report.to_csv(reports_dir / f\"{VERSION}_candidate_report.csv\", index=False)\n\n    summary = {\n        \"version\": VERSION,\n        \"data_root\": str(data_root),\n        \"sam_manifest\": str(sam_manifest) if sam_manifest else None,\n        \"manifest_info\": manifest_info,\n        \"selected_species\": selected_species,\n        \"profiles\": args.profiles,\n        \"outputs\": {\n            \"reports_dir\": str(reports_dir),\n            \"submissions_dir\": str(submissions_dir),\n            \"visualizations_dir\": str(visual_dir),\n        },\n        \"extraction_report\": extraction_report.to_dict(\"records\"),\n        \"validation_report\": validation_report.to_dict(\"records\"),\n        \"candidate_report\": candidate_report.to_dict(\"records\"),\n    }\n    (reports_dir / f\"{VERSION}_summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    update_experiment_log(output_root, summary)\n    print(\"\\nCandidate report:\")\n    print(candidate_report.to_string(index=False))\n    if not validation_report.empty:\n        print(\"\\nTrain validation report:\")\n        print(validation_report.to_string(index=False))\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")

print("scripts written")
        

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cv2") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)
else:
    print("OpenCV already available")
        

In [ ]:
from pathlib import Path
import subprocess
import sys

# Leave DATA_ROOT and SAM_MANIFEST as None unless Kaggle has duplicate mounted inputs.
DATA_ROOT = None
SAM_MANIFEST = None

OUTPUT_ROOT = Path("/kaggle/working/animalclef_anatomy_template_registration_v20260426")
PROFILES = ["strict", "balanced", "aggressive", "force"]

# Fast enough for CPU while still doing real template matching.
MAX_SIDE = 700
MAX_TRAIN_IMAGES_PER_SPECIES = 500
MAX_TRAIN_PER_IDENTITY = 8
TRAIN_PAIR_BUDGET_PER_SPECIES = 3500
TEST_PAIR_BUDGET_PER_SPECIES = 45000
TOP_K_SCALE = 1.0
SAVE_VISUALIZATIONS = True

cmd = [
    sys.executable,
    "species_anatomy_template_registration_v20260426.py",
    "--output-root", str(OUTPUT_ROOT),
    "--max-side", str(MAX_SIDE),
    "--max-train-images-per-species", str(MAX_TRAIN_IMAGES_PER_SPECIES),
    "--max-train-per-identity", str(MAX_TRAIN_PER_IDENTITY),
    "--train-pair-budget-per-species", str(TRAIN_PAIR_BUDGET_PER_SPECIES),
    "--test-pair-budget-per-species", str(TEST_PAIR_BUDGET_PER_SPECIES),
    "--top-k-scale", str(TOP_K_SCALE),
    "--profiles", *PROFILES,
]
if DATA_ROOT:
    cmd.extend(["--data-root", str(DATA_ROOT)])
if SAM_MANIFEST:
    cmd.extend(["--sam-manifest", str(SAM_MANIFEST)])
if SAVE_VISUALIZATIONS:
    cmd.append("--save-visualizations")

print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)
        

In [ ]:
from pathlib import Path
import json
import pandas as pd

out = Path("/kaggle/working/animalclef_anatomy_template_registration_v20260426")
reports = out / "reports"

candidate = pd.read_csv(reports / "species_anatomy_template_registration_v20260426_candidate_report.csv")
extract = pd.read_csv(reports / "species_anatomy_template_registration_v20260426_extraction_report.csv")
validation_path = reports / "species_anatomy_template_registration_v20260426_train_validation_report.csv"
validation = pd.read_csv(validation_path) if validation_path.exists() else pd.DataFrame()

display(candidate)
display(extract)
display(validation)

summary = json.loads((reports / "species_anatomy_template_registration_v20260426_summary.json").read_text())
print(json.dumps(summary["manifest_info"], indent=2))
        

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage

viz = Path("/kaggle/working/animalclef_anatomy_template_registration_v20260426/visualizations")
for path in sorted(viz.glob("*.jpg")):
    print(path.name)
    display(IPImage(filename=str(path)))
        

In [ ]:
from pathlib import Path

sub_dir = Path("/kaggle/working/animalclef_anatomy_template_registration_v20260426/submissions")
print("Submission files:")
for path in sorted(sub_dir.glob("*.csv")):
    print(path)

print("\nFirst file to inspect/consider after visual sanity check:")
print(sub_dir / "submission_species_anatomy_template_registration_v20260426_balanced.csv")
print("\nStrict is lower-risk, aggressive/force are leaderboard gambles.")
        